# AP Model 3D Foto MAX: usa toda la GPU y la RAM (A100 / L4 / T4)

Foto → modelo 3D con textura HD, **esqueleto automático compatible con Mixamo**,
**pesos sin huecos**, **6 animaciones**, piezas por hueso y exportación a
**Blender (.blend) / FBX / GLB animado / OBJ / STL / video MP4**.

**Qué hace distinto esta versión**
* Se ajusta sola al equipo. Con una **A100 de 80 GB + 167 GB de RAM** usa el perfil **MÁXIMO**: forma octree 512, hasta 150 000 caras (antes ~25 000), textura de 4096 px y video de 1080p.
* **Usa la RAM a fondo**: carga en la RAM los pesos de los modelos (cerca de 30 GB) y deja **todos los modelos cargados a la vez** en la GPU (1 foto, varias vistas y textura). Trabaja en `/dev/shm`, un disco en RAM, guarda en caché los resultados y deja que Blender corra **en paralelo** con la siguiente generación.
* **Pesos sin manchas**: los calcula Blender (método de calor) sobre una copia cerrada de la malla y los pasa al modelo real. Cada hueso mueve su zona completa, no solo una banda.
* **Malla reparada**: quita trozos sueltos, cierra agujeros y orienta las caras.
* **Cero vértices sin peso**: si alguno queda suelto, copia el peso de su vecino; el informe dice cuántos vértices mueve cada hueso.
* **LOD para juegos**: además del modelo completo, versiones al 50 % y 15 % con el mismo esqueleto y animaciones.
* **Visor web** (`visor.html` en el ZIP): botones para cada animación, cambio de calidad y modo AR en el celular.
* **Multivista**: si subes espalda o perfiles (o una hoja de vistas), usa Hunyuan3D-2mv y pinta también la espalda.

## Cómo usarlo
1. *Entorno de ejecución › Cambiar tipo de entorno de ejecución* → **GPU (A100 recomendado)** y **RAM alta**.
2. Ejecuta las celdas **1, 2 y 3** en orden. La primera vez tardan unos 6–10 min: la 2 compila Hunyuan3D-2 y la 3 descarga los modelos y los carga en la RAM.
3. Abre la **4. Estudio** dentro del notebook, o la **5. Página web** con contraseña para usarla desde el celular.
4. Para no perder tus modelos, usa la **6. Guardar en Drive**.

> **Aviso "No se ha podido completar el guardado automático"**: aparece cuando el notebook está abierto en dos pestañas o dispositivos a la vez. Ciérralo en los demás y usa *Archivo › Guardar una copia en Drive*. No afecta a lo que se está ejecutando.

## 1. Proyecto (código incluido: no necesita Drive ni subir nada)

In [ ]:
# @title 1. Proyecto (código incluido: no necesita Drive ni subir nada)
import base64, json, os, subprocess, sys, zlib
APP = "/content/ap3d_app"
os.makedirs(f"{APP}/ap3d", exist_ok=True)
_FILES = json.loads(r"""{"__init__.py": "eNoDAAAAAAE=", "config.py": "eNq9Vl9v2zYQf9enIJSHyq0gy7LieB40IP/WBUiWokuAAUEgMBJtq5FEjqTcdEM/1973yXZHUrHcJun6MgG2KPL4u/vdHY/n+/4J06zQlNSUvH13TT7h4P3hBQxYXa0Y/BPB5LKqSUMVobXm5I+OkYLescjzjnlLupaSw0kck5KReUzeHsHeyewABzCDWIqRTtEB1MXh72cXlwtP85IrUsOv4SXDN4gCfElb0CBXFJdZ2xtnrNywP0OzRTDFlQcbhGRGuHWiqDIoaLFmaICgq6oF09+Qcck2Y7VuRpZlQ+uaAhT1NHvQnaSGM9lUJeNE0RrAjEIJeuquqICqcwGNPN/3vaoRXGoCRriRWne6qr3+Uyj7+f7y8opkIBexdlNJ3kYrpgP/8N30JMc1PyT+uOCtZq0eUzEt/ZF3eX2Vn5y9f24bLH+1Kwebq5Iq2H10fvrryemzu93yDsId8C2Z7N8A4nklW5J8JbpgtPAIPFp+sgN8HEnNZbHeTi7tRFR0JY0qldMNrWoKoD1G/wgwbiAKxuUQnqpguZAc0kRXTAXxaGePZBCmloiopQ0L4a25pnXesIbLT2RMktevp7HZwR4KJjQ5NS+I3Fa3oEp5AzD/+N01OCKOYkdY0iYvK3UfNFWbLyVj+eoumzvrIezHVAoGBwZSS0t6Rz9wTDuTc9sEUxVZU0gnJSgkzo/43fKQAG7BI8ydr9yJmsAlNoUiNCCHM7Nigd+j+qPICA15Op+b+Z8yMjB519slQD8C2Wxxxvs7cpAuDb1nZSVVUIbgxkrpnN9nV7JjT8YChFDhf3V6aVNSUL2OPvCqDTD/IQ17Y6yOb1ux1b4NXGkqWZ9nNkU2EE1QaZPY7jQT9mhGcDB095hBwcgm1I6HC9EpazSM8oJ3LaggXJLELH/k8j6EqDW5C+Bj+sABctExRkB0DuKFdceerZe2WI7JL/DhDc9FWQEPWyoz39ZKSFFeaFCR7U8SUKiZUNl+HJJi3bX3KktjfMKdENm8ogVT2WTfLBOodCD6wywED+Ixz5I4nYdQ0+qct+gl4+MnYPhyWXNaZj/TWqFfsUjmEpHjOeAq2ogavmZpaFzCpMqm1uFwjWxdMJ0vBg5I0QHPMz88vxrwTtHQr3gn3+Cd/D+8EaznnTzBO4kd7/OUJOmLrC9OT84GtKfztKedbmnPXmQ935K2PJ8jbVl9B+uDZEB6mjxBWrHFF2B75Colk/0XSR8d/nZ2PGCd7M961tMt6/lLpNNnSE/iJP0u0iYTnuM8mW05T9wR3yPYBjVdsaZ9u7PqqDRNDBcV9AxwVwybHLgv4NaBqgg9R8sdxobXGyax5WBMohTI2gsD+hLscphsKriqTRPi6qVBPrI3toMBAcStQREJ3JLpvlDjG7QvBJ1gWCMADd59dzWKDIK48bGEmfYpNwYr/9bWNUzkWdxLQdeF7so/smq11jtCadyXPjcBx3BwGdz4zoNmT0MfguFUSFzhEFEnSqpZgEHblnO8jiWU4TLAr5BMRiHZme5ndyOMdTzDPxs/qNEy2y3eW4R+BmF27hvxeNWoQlZ3LBDutnHrwdLHPvUvcfMKjH51+5kEOHZm4/fboxH552+TJrgyXMD53dt46aMMGo0SGEAQcS00rtghrsG8PTlm3g7d/JeQa6qge0Exc25e3S7Cz9hsQ1aBdN8K4zqMEUM8PAUz6H1QtvcpbIBL/F8appwq", "engine.py": "eNqtW+ty3LaS/j9PgWXKdch4RI9kn+Qc7WGqlFh2VCvbWsdJdqNVsTBDcAYWbyHIkUYuP9f+3yfb7gZIgpcZ20lciYYkgEaj0f31haDjOK/yKi9PWQw/rvLY0Xcs5UnC2dPnzP2xznY1z54+PzphT9hJuqX2StxXdcnx8vtEZJEo/dnsZ5WzSLBSrOpS5ep0xtjX7F0e5Yol8H+aRwJ/f69FxDO24uWaY5vIGMz18upnxvFiKx6YW4gylgl7dfZfF6/esCdAibGzy3dvvDnjSrIsZxu+Y0JBP44UqlKwOuO0BLZDMkquawkNwic2LmGiQgBXTAlWAIs4e2bmfnv2irkrvtoI5L/ga5kBVbi8lFl977FE7GCNOXGBK+DlaiO3mvWClzyBdZ0yQ7QERsxaWcXLCFgR6xqGK83JecKqki/5e2gWaZHDcLblSOpJJLZP1CZlbiTVKsdHwJlnFqBFRNznlVgLtsoznGrFswjk+O/NRjD3B+j3mIZCnxIkIzJivWGVRtoiYmuRgSBXMs+63ejkVgpVJxXtFUy/rnFR1BElrsWmWWWwGtgYtUHZJc1udGJ7X6tKKJRUISpZYpdUqtR0FIrJTFU8g/9E7s8cx5lJFFCFDINOlcCrH9egeNAVNmgVN+3rVXOFsydy2dzKvLl6r/Ksuc5Vc6U2dSWT9q5egnBXQrXt1aYUPJLZun0gU9FcP8gClFTMmvusTosdMpYVs7jMU3Z1cclM20XK12Kuf94UaqY7+KxbYCzXs9mP52/PWQAM+gWvNn4ky4ynwm3u+VLhrxuGOHEYet4Mdiq8fPPDf8Colln/Ml/dut7sl4vzX3+CBteB2bLKmTNnyVe3+JuImO5Lud5UjjebzVYJV4qdZ6D84pT2LBIxC0OZySoMXSWSeI66Fwev8wyWkuTroChlVnm6N/7DTn4BM2I/Bsqg1+VHohKrCjjqdQQK0BX+9h+XIl1iA07Tb1EbXoj9LWG6nWwsOHA52aKVN2AfPg5Z4NEOnr/giRoMWWorC4s8T6DHKvbfkdiv4P78HqAPoNRN+X14l5e3olSBZuHaMffOTScF2NaU3wrYZeUaSb35+V34/OLtnIl7qaowvw3elbWA/cH+X7GjP/FPI267swCCSc6j8E6gCiizwWCauQpcpxLZCsztiQX/qC4Tj9Ot41kqAFZ7KQSrWthvwRKhlRAYcQju3vMyaeEf0MNHg2/IkHls6jVo4zrmKxFu6mVjLAogL6StC0F03RAwCQVbcn3TPqrKXccZ9QGdLAFmBjRcz6eF9zvjPxmzktpCGeG4Pd1a2mJLvWDIVirA0z19O34fA8OqKt3YJ5NG8/aIVKyn29Jz1a1J3K9EUbFz+kHERgfYn6WxLtd5nZPHqKMcnJggxG393I8vTmFLhdcbWgqA14wtZu1T1JUycomx/jQZCHvRe3Inqw3LC5FRd4SXJUyxrONYAFKsg4WH3MZjmdxtYJEMVX1aXkuYSpul+80z9q9/sZOFN9kR9ivLK7bcL3azwmyyQ4b7ARbuLr1OAogdiPs+/rFAjFY7DQDH39BKxX2fjyqvOKKGqlNX3IPxF24ZzbUmeGN0dGPnSluMiS1MuATGcso+aGIQk3399dPFqX8cf2Qvv8dmZ7y02PlgrYAdserUX8QfldNNauRCVGctSiBEGGi4A88MEEuAZO4IWTVCdQttPGUOht/JEO989D4An8pf1RGH5VdpnfgQauZ3YRU/PQHJIK0Dg7LsS7svAbE2KS9vh701wuyeRhD8GJ9jOP8eKKzLHGK2tyLNt6KcHkM+By6aYS0mPpfvXgCPr3gFyJetr2QhEnCqn1YoUF7LA0pFHmvCtBsPOWJ0gpb2mftpEUz8QNEwgIQd7buAQimfs2MKzzzf9x1vPL7xyZ9avY+yC0GRIfiF28gdKekenwNBWZwn4HIDZ2MaoqNIVkfbk6OFMx+RqZUIFY8h4shUDv5X+09LLEaPGSx4EDz8ASnBqEZOW15KsHnA/YqrCXGNnNFU9PLnxbhPlOCnDwsTO0xSOyzRz/RJfVEyzCQxjYNMp8gzCVHVHD1VrTA2UDkkKVrtBi6q2T4d0rU7qG+/fPuuaJxrstnP3bMeCsDYKQwgyq3lTwqiCUunx4w2etI8vGnSoRJVqFcFWycfhNtEofDQjkCHaAE98jhG0Hdupl1ox7ovMg77FlKuG66KOjRD3T+tHO9MeeHirK8jDNYB9FDUnLLYFWhKOeXwWkuATGQHuQe4Tisp9SYin8PpgokrGiGBEwrzLFwXtXMzDSTYe6yNX1ESDxa94qdMJE2dANxgJQk4TE2ji4pne9HCr3LXAbE73nTy0vN2VkjxytRhEphPxxOfjA2a0WEqUtfrooNpPQNLhks7Iog/aaY6+jgQuI9tDrJjjCvBJCTpYUmJmSj9VKiNuWvs8hU8ektPZnuU2eRf9mIwUIOfT4zQE7UDjILop87NvsGGvcDizAWJclAEYB0AsEZTCYbk5vs1ffzPXktAWzLmBndvuM4/mmGseLqUvES9rnjKM7I4g63G4Ky4UisTErI0ZbTvZv8KRXWaiZZ+mIn/tikWIGiAv5VlVfMEp8rL3QCYMITSMaMOR6lPiPHlileAuF4TXKMCWx2lCvmWywS1DjqJBASwmBZP7GBZDLxaxNmHbeqDK416IfsTfDqI5Bc6kv+//90HbLGDKPJhbSgYJfnbtuRpuF7+7QaHW9Y73M++hAuu1F9SXpBY28qEsisMhQEEma6xtoBBarhcjzKGGDscAoYRGENyTbXClXCJNqTQ3lBzcH+p4OZTSgoPrECia31TKF/cyzgET5uBp1Gi33XDYb+TYsNhADT4iNksCJjz9uX3Zw6hf1b4vCz5jgZeQxgxZ09v/FRmoB6Qq/59YbPeikEPzatuhqkVdBE/UkfUgUC/cnF2xxsuaNjjzFKDr7BCjZ6zZKBt+fK9oFo5xhQlWq2uJtc8KtFZoneFtGktOlBGAUyutO2xU3N2r3S3LM8eRJm7nH3Hjhe9EA4T7Hs12K/7BYw9nrMd/O6Ogca90hKc0xW/x6td+2xnnk1IjIRQ5oXrIs0d0WWP2THRhV9LandztjEjehCIawCYxkKei3089jU79o//YbnE31vtysSdEfYcAhTQRWDPhVnNf3Zi/zvAv6qMzrquAm97BzADpo9D4W6j77xRXg5DwS+gi3WPFyfPIC2Dv56pKPuXZ69/+O3NTx65WhL4yTMNTOr3DnRVkcgqVBshqsZR4/Wc5SWlBFQlnmNpeI5Z9NwUh3vFvecQp0T6Zcsmf491PJPx6JchmcCilhJY48NqBb4RiHOIunvVPeQSZz5o8lZRtlwved8U9lluH4b1HDp0sxDDrBoxQwvJQglq87y9dtYzAuSrtQISe9tvReVh7vNs5y7sSGoNGgKcgBCRA15WVLKc61rz3HIkWASUkFZiHVBkdQphIWgO0B0CXcy2LZRoyhN1zMGMuogj+4F6YkSlyR0mNaqO2yv0eQGyjEDBcTqYx+vZ/xTlqWGIE7hea/SvJFQKga+Pb3piRTkqkpqiCi8+QkWDfmBY6nqBO/QrGPLCXzy7GeIRdiewPiWoBGVhmxrwUp1ivbrCl1NriCSMe7NLoDiaDMgn+3Kdua0vNncuQooEDn4FZ595c4IYeICY1D7V1V9J1V+erYWbeR2z+DaISOU+aK8sXN07x95DHrpReV31X3LgGCQFoKNh10MCD0CPJpgTxwM1AyLX2HrTRrno3tECLLA1GE5PEaxA9p5ndHsEaUDxLwk8qP7S5STAFpelATjMBeAvvw/xLYK9poO1PPe50O9EK/ECxpnyGi4Eb6J6BTefGYq/gJS4EqUh0QkBOUMj6jW7novPR70m2Rl13hs977AfxKu9xhQVqW3yMRH5SVQDj5qCUUZRiJ3cfl+a20eMlCtUGbrVQh6RwDaZrUMT+kQ1KCnG2GEz3D0wBHczhGgihJRCYvUqFLBnB4asILsV4SYHY0UHTheYAp0sFosD84R5iW/DtaKEq3xDuWUyzBowu0ixukzvo7VkvMnMpJTY2B+st9M0+e/0r5uSHMU9LBGaKNJJSZbWA/NmOhjY0hcla6zb7+niW8PZRH1kshR2YKnNckHCEEXu22lNWUtDb7Ufy3vYbjDpRA30e++AJGm2e9R/Uj4nh8tPJYUvdBwiT2UlI52+noyCWEvn2XcdypxO7bkFHcZwO1wC7x50GDXESRJtF8RR+QfRy0DcVoo7hOxKFPCTr6pS0H6KaM7WNTAPIc+nXs9QAQp9xO0peH90D7cm8iDqvqxEClunYw2rutW5FCwSUwV7byWMggqSGk6GAjtuRxeyEOOxcUNVR7Jt4+dU5QbOq9pQtcV+JbKPOjwaxld6+N6qXjvDREmuWZ1uqiM+rDB3dQWRFtVOv5R2h4md3qABz7Q31+ZwR+fuu7rGS+00cqwtbcH6AsMBpE4ZlkVQSSgowQtv8GZTk5AZ1dhWus7repO6jQt0Kf8PKLMBfQ67gaSZAU2DV95+z6mV1y6A4Sj91NNkV5s6u+1OVejbQ4WxxgRCtYLwLcAKdeW2djE3Z59gscH6AGN1VdRVWO0KkKGBH8eDkHLKVudmk/6S4MbUz1rzN0W9XnhjEAAdyB+tvBpK/ZBexi1N9m8BSKriVdVEVg6WDgEWkaI34XDGteGWP1u3KZrtK/INBMTm0e2NBiNEInff4SVCJYIp4NC99Q6c//iK/aLf0pVCaydeu3FJp+AeY/GdJ2CJvu9BBqBPI5Z4jIMlOeMR+AIxjInbaJhTvYCk2FqCahwFXhPiacPFW1CeP+zFm+OXWKTpvXZEr058gnrP9Xs0vbgJh/5J5nscdjH7XL9Z+EuUOzenKlGzb8Wu79HyorIDdgQac7wPgfrYwshWR+jEW19u6JNs7RjnzgdxfePXRYSJ960P6kIQ6O3vhC9llrsKw5zr01P2z28tCbad8CiiH9VpoVxcIu5TWYWwfPNqdWIiI/2NvxH3kYTot8IJjk9uurigrDMjPl3GYWgtzVk9MBhzSTZjrqnWoa/HdZPwYFFoWM0dECDE15No9DY3FOCYa1Dihg/wPsHxydNnfSoNSAd/9xfEOL60xvcKMhLAwIYOrgXH/j+gMQdoEgFk9n0SWKvKA+cHnsqMl85cP0AH00zN0yLpWAIhV1jWn1iTgTJz/KUUiRRbnPEbCs3XJcbmCU+XEWcIzKdDYLRjFXpFN2nevbenV619znqSNcVJfQ2qf3Xt0HXvbCGJ3fQ0N7qrvrH70q6Yrvpa96Rru6N+WYTd8Ep3GrxRbgVsOnb3unt7bw8yu9Csy9yZlem7HhuLwQGaLo40O+GCJpyAh/jPWlb0zp9qkI7NJiACBrzG6ZxaNQ1jP62Ce825WbsT2YLdZxw7aD9lDyIr6hPW5mR3oid2r4+TddNpyLJDdx3S2jVfU/fszNtrIvvJk482bE4HDz1xorcGrJ0NKbSOfU8AwiW4xF94UovzsoRY1YEopOLtAfJIGB9mbR/iJswVyZUxg6CX/gRNFqTxhv5qwIH/Dd60Ye901NeCT3OhEQj+b5FH/7Tgo3/me+SDOER/bQxqrzogMr/TZDp8WuZ54ra33ryFJ92gb7wOp3TYa+4sx3LbQAx6X9vv2rt422qTPjyKiVxzJF2qyBTbdCMowbUDj8Be98Uvb5vvCfCVAQ3C6OR28iBqj3Bn53iY2jon/z6XmXulz1iHNHmPnn3OGruMTlf36qJTie/pF0YNW1/xbXdun/jTM8cORWrhh+yjX2RrLBX3067mQP/gLa0FbODvHJ3ZIbKRmZQpfa2D8eDFmcMeTycyEDq6/VNqDkFKnbr9TB3lYAlhi7YJUrAiWGdY4yZ7nDrU2KSJlBVFoxoBFTI+q4TRny5sDh4Ozq6ofWJ7BjGEcynTQpLY8MVnqSXI1/V7UebK8aZyW53MmKKyXsigTNNWuvTJpKldd3i5kknC/XWyHErO5hG39kpmVbOzdqB/cdZmKrsmU/GcL9gFgwog0/y2OfeBi2tSymEy6ZkEEBv1rrtWljjcD03w4G40DBwSlD4RZRJH+QAoMSWzg8XI0YmQz6qzfPqkRP+0hE5/mo+u4lrQRw1J8z3WafNRG33uxiBpKYX+8qH35RV6uKno5VvUVkP8FHYbqCSiAudBn1DMAYIhT8MKJcRJO+1RbPdIqV1PvMPvS2ijVFXGtFnOo/9+lD6Kwkc/Pnr16KcQcurH+wCUMsB9+NmllPZ3Mr6ql5CTumN/YHrN2Zeowpyy0L636Nxy44+Nn7XcbetkrUC/9ZDdIpb6uyRYi6+/fLMDTP3BmL/Ki11fffHLLeAZMrM6krm/qdLEcNp3L7EuZdh4cGCpsA1VDSyPj/kO/Yj2XW0OPHS+5WqoEMYdjfLhphNtsXJh5ERwaIsBegzWKbM4779wlMVwelIjjH6xUJj7mBM7vaBjwIcshk5YT0LJNOVN+pBP4fX5aJLu7ohlYPxQEzPxUPJAI+K8/VYzJJ8a0OF61zibOTveEyy2owzFdhzRhFFWDzzr1bT3wXKxfwLz6WSg6wX669gAAh59heHOuqjxAdW+Owm0pQYXZTFnRkYg+TuH3j9HgEIBZEoiU1in42olZWCOIpjziAGek7Aib3zdPd5JY8kY4SFlpO/flRA+uWhP3hTG/fNb6HeeFhzATXu83y6uLCV4IKUBSHnMHP9BFk4/UDKfX/q/yeIF/LoPelnzruHiKnx+/uLy7N35c/oc52GcK5E5Yv1FRLgqPBOL8Syq84TaP5gljdcfez1bMvZRphjJGFOT6ywHEQvMcYaH6HW+TOkMTB8QSVhGgGvCnQv09kGQHtx6E98w6tSrtE4nDI7sDsJq7NlWjszWNUf1skID7J8A1dn4o6u2XGKJdZVGWP01run7y/PXz8/BNTlHhIlHRxBngdveHdFxkLqgh1T4XdDl1fh7hSlIbjwRLMcvdqScR0d6Apk5ZsFwA2t2JlwLJvJHRzDYMRKBOy0Uh44QufpGk9VSMg36xpsipx12K83mCdVF9OD2XhNuqiC60dxNkm53AZd4TAF++8jE7lp8R2ZTGm6NH7ReadCXas0n0T4WGGHH5hDhFFTU1y9FTEEMka+5BESDtuDpN/b7dtDF6h7DA+d/MkdvUEImmJBHLCEYiYyxQwf4oSdgLZ6uYeBHEVNvj/veyzmjI0QsLnkqHHMkCqZozkdBLILKpBBBXOcFHkzun03CTzdRXliBxZceiz1ZrF7O9dHTxWJxOq6T6+bZbBaev3558fq8+Z5gNkOLW4sqFPSltTv8jHqd5EueMDNsZrhqqIwKKB35845eC7iGmYbY/wNtZgVo", "blender_rig.py": "eNrFfWlz20iy4Hf+Cjw43jZgkzCpy27adIxsy8cb2dJI6l53KxQIkARJWCCABkDq8PP+9s2jLhyUZPfs244ZiwCqsrKysjKzqjKzbNs+neRRVlrT0Hodh8k0zC1nx9uyDs9OrdRaBoWVrMJ16lp/rUKrCK3wazhZlYFVRIm1DpMySIJhp2PBf2NRvTe2eqXVt3rH8pWfR3Mvu7F6Pfgf1Fum0zBOvXkMJXvpqrQmQZ6FAPTc87yLTudDMAmHAHPgWR+XWZrDlzgAXGL4twyvy1Ue3QbToItvV0lIf7M0CbETWQQ4JlYYW8UKGukSatYEMM2hinVjEawgBiBQKZxG8NIZeM/71hJg5ABiFk7K1PWg4pZnvQ1LeMQ6hZXlKZSYRNBSAeVia7IK8yy1nEkAnQR0FulynKdFF1BN8A/gUriMwDKaRkCLlFAFOJHoTxcwmuQh9sMKC6BxHJapNUkTK0Fg1JD1KboOlikDcpb0ABQdfogyaARI5lpZAL2hEYrydZAQukESLQOFrQAi6PFbEpU3XfgDTceAghh67PS2Zx2HRVoMcbQnQTxZxQCwSAEZQDIA3BD5PCT0iaKTMCfaOuv0OoxdxJ5bARItoTvQ6ymBIvIqNrvBBmBUkmIWhTmMmTHKiJZn7RcRA0pSa4bvExqiYL76GgKdsWlsd7ECfK0lsCn0f2XdpoTlMouRpRyoW6SxoB52YBwkyDtQIgW6FcEY2CEpYaQ86zSyAhiIoIi4PRjS4AaQKbIghl5A4wxmGhXA+BOgA+AMiBAGBVJvx7Pe4HCaxHfyMEuLFJoMllES5Nh2nofwtwji1TTgHyX+HQdRHOTEfLuedXDNzP/+8DUDnAKQd6+/dK3Xhwef3wLjAZ/whIDxDpdZACxQAkkKt2sdvf4vMdinZ4fEi7fIw0A/wrZrHR69Law4mofEsuuogE9X4Riouo6mIXdnz7NOQmKudZiX0QR6gxM/Q4IjYRi/yQoICGRQZXgs9OB4Hdu2OxHNZSvI5wCxCOXz1wL4RfxeBuVC/k4L+au4UT/LaBl25MM4u5E/k9USBAxKq6wzy9MlQVqVUVxYosSnoMyj6671r1VQhnkCQ9O1fsepnnc6Z31rRKA9/MdxO50OyAEY2rnzOHCHRMYsj5LSmdnnMPWsb0Zhq2ed9Yd73mD2vbiwu9ZjYJtZvCoWo7N8FSKsRyDm/uZ/SLWiA/+sAVOgh4c/z+UPL4IZde3YvZ7tWk+swfDCimYWPlowWrIUzEeYcecXnSADIHIYvP18vlqCfDzGpxw6H2ReMJ36gXiPYKMEOjYNi3JkR0kGv/Pwr1WUh1PRx5YqINcfVA7ISbBnwSoG8IsVCk9gQFtwr/HfZJEie43O7WBVplDLLG3ZwIbTfAUynT4lUTJfJal90drqIozmC0SwvMnC0SxOg1IjAeqgHdW0BN7ZUKnv9ZsYL8I4G9nzKE8tmKYwjUDMzUFYwmQJYiAL4Eoy8U+7tUGaiCZ13rAEuaO0D9JKYggMa3Sq/7zfWq0IUFS2V9ppJwQIHND8oN3bW2qtE6dp1t7IdjuxwzgCKbKR2nubqD0DrXwbkIJeBhn+sAQsVAY5KBgX5C0IJ1JLyqQIXlgoBVC8yaZb8ZoFRdnajX5XIJABq48DUKA0ICjsgVldnLwFzrvMo3mHQAsH56XbAWG7DC7BGMnpTeHB5OkCx4Ca8dNLMXc+fn53BPW/2ThjhiQQPPgJrB6g8C7g3fnF906nAGMnhIIgHT1QENg7j9511nHt9ToKr/w4uAlzruXlbAbMMsR0u/+3ZRdRJMwLEqcgP8C8mZQ+qFZHSNUZ8H9KQoraT8dgXpbFUA1t6ok6RVg67wIQYFI4B5MyWsN0dECVpeUCGhk5roBabUk1dI0NcVndwrXZAlO60bB+vY4ljh4hgIROBUazySpfh4Wfzhz4JlABxfcGbJGgjGC6kGEn7R+0srEgaCJPVHVBqe54O0+e7nrXaFaCeeJ6qDoRUpnfaKzzEOcgqO+iRBgKABUIrychmPT7JWi88aoMD/I8zXVdgFpwH8H0HqFCkAQiVkAizcGMgNoIGviLXuPcPb9wNRisUEALGVagIh49GrSVxSZjEyiVQnm9CJIkjMfBvAFbDT9g+GTE3ZyMq70UJIAigv5ofhR+kjkwQgLWksZn/NWbBmVAbyYpvEkyD6yl8saBoXBAkUvDxbUeW9swo2lyQyESOns7roClynnQqTCYLHzokmNPUERPUlfDn6Qwk4pFkIVOb9C1tvnTJ245AGv5BnGEOY8miX+V5vG00icA8g/r0/kQcBluX3hnoNP5afvi709JtoeCvIOSAGSyd7UEbIOpPwvQGrpBji9BeRbOCqQU0UlIoH+nZIFCODQFCMNy4RVZHGE9ln5gYrjngwsvTq/IIAFbBosDCzk2rhpRvcPfcmaLYZY94a753CIWcGZRHGILIwW4A8Kd4Y1GAAbGwa4CAXLAS59h3Q9gNr6278ACPrfDKEKulQcRWGWnN0UZLg+uI+CnGSopWAXSyoXGapoOLRuYABoF8bcE1gqRjOfpBvmJ1l/qIRcTjp8OTj/YF0jGJAUbm6pvahy0ZpBPFtGa2i8jHHFakxWgwlRrJgwtiN0KGRgXVHagOf1JHAa5QxPLfnN4sH/i//Pg4Ng/O9n/fPru6OQTKkgJiKGf9y+6oqFzsGiJDXi+4ivXegV2bluDX9MoAaaBJyBRQ17jey8JSDDYn2gjwtb9IkFTIaXgMCSoFRVEQPiiJRU2jsJFNQMr43RNWmma+qskjhKpwDWhQD51aljzQhhG3g+yLAbJlE4CVA1UF+xoND31YwGr6VCuLx5Zn8msiW6DfCj2QLpyyyPlbQg0PP9EE6Wy8wH0jaBdsIQCq/cHEpitCrJzuY9SeGoEHV5JeScCJQeXWh40FUEBxwAAi9AdmKlg27odEosVAd1ZwgJsec3SEqxap+926WdwDT87C/iAP5fX51sXsMRaJvC3aw3C3h4IIl6/FB6b8dZTa9GZwNpucgMfHCjav4DZAlX7Fy583OrSy4F4OeCXnc1dO0XyOrAq3nFREPPLMywVc5edHjbXm9zAP4SZ67oa3iqb4jA3xhg0wjT0i2UKNgisGwU/q2pZGt/M0wRY+6W108f/eMX2OQUJ20LAD/CGNJQzSc+HXWvrgojnsqkosZA6a1RpS2k83JXIg9pXhUnVxmbe8ZejPF0lU+cDaja3g6tkm0QEyGVs+dyW0G0YMf0gv1KD9Il/uZrxYGmNAouWd8x/j4D8BZgX+Ty1HODtGUqT0MIdnjjgXUlAK7V6r3iLJKa9Jljj57i7UwZs6XD9KrkGglzAXPoFig4WYxqhysqSRo2ggfzx+rtgMXzgYTIWowSAO4vG+oVkV3hgdJBkersPuwuTKJqkQyCSLEkz+xNuVkRBDPZ+Ec6B+rnciF2HtJp8cwOStQB+PoAVSwh/aSMnLEimFXFKalMYGwTJx5dCdC9xwQOP6qOUdUsg5tRaemgBJCAki6rZl5AO8PCLX+Zh6NXKCCiJVkGvT9++849PPn5+8/H48OCt3TT1Kuat+V+CGnNVFuf2SbqaL5KwAObxxKrLXwfxCmU5LAh3G9WFIXxAf2DetjdA9vBPmVXAa2U4wW0+5jrcDhZLHdw29iVHAMd1rQ9ClYDgALFxS4IP2Q7VnGRA+WvrQjD/AQjxEAR4ngRDK1slJe/JE8vjUpW3S0M0wGFqBMnXQAj+2BoHX1MkywcCxZY6temgETounGsUNH2vv0VM7Fr/y3Ju6c3elvHiFbwY8AtGagLSfbJQk4kAS/FD00cKNvri8uToezvPEYZkMNSjDrwkwNbLkYT6EkdylxHSo6WalGDkh0fWclXgDuxXJA0sufUevUXED8CCAMVEkBZR5hPZGdoT7PsOISWIfQyKktdCYVgyrZgg/T0sxt2PZ9h1KHBO/9AAIpH4a9729aX8Orv2Y0U4GAXCLXHiGRfUpItnim79XxXdoHreUj1vVM9l9V6lfpmG/o0CIBEc1McNP+j6PPZMwDCMNwAgvdMCANrfNkm8n0wWct+F2bpyIMMf+DyFbTuUQ8y1yIpi6HpqTBTX8hfBonE496+JXfrPd022Q+wQpMFbY1q6wLtz+keNp9cf6EE3q8e1dapsrD4s41gOSwUnpDWSi5672A5zO/4cEDu6klhnaV6A5R8QzfhQKwvhN09DMIJLgzR971m/No+fPVcvqlN+a8+Yz8Wiij6WI+ACfTm4j6Hi874kA5VQgzwYKCoLcLKb+Ei9fK57ya3L0rc0UM8HuwafQr8NRtM0VXgNDG6votLvKLshQYmARFNyMw6/wlsr4LOJGPcAUVqSfOp/sG7g76+7H8Qe0sxaAEc4BWh0UxZVaL5Tp/mvWphiRXi4hoJElCeaaxtcNcGB2epXOUvsBvAphSPhOXVgXUkwJO4uIwQvabdMtwNo4wEGbUDMaWQY3ESwPAKkMZiwMAdT4oNgXRoT1h7nkUY9gRUdvQOYsMwJ4rmH+6vOBOZniTs8wXVUjAZCy+xWZxNCQwAwsgHa/p0NnWYOyNCS71ryYWA+kPHdtXAxxEO/iLtWeonTmsZvwLAXOb7N5dueeC11EVZAWQNFjMFmG07sr154sDQLoa49zoNbUDpZOBdb+UL305npqghyUD92q62h/7Mrx8khHihOaWffIMI0mpTOhxEY2izeRvyny6psRP92SauM8B/6mePPvLk5TsJ/RP92hSQf8Z8uy6aRkEjIXiOatg0YOFvx061iuZFiPaQq4IHEp5/5aJHjXu3ro88Hp7TNqOE8wp14PN7G1cc4vKXjWDxuzcDOVju80nryx0giRwrtD7j5DgCn5/YHXEXAX4GEzex1jCa7PibngVjcchWimShYIBOeL6A38PWJ0rXqcfC88rj1TLNwEk4upeTaMxRkMJVvf9Xy7DU2o4jp2HhwD4a+08eeUAd4XG2ehax4XF2gEByPC0FjSebYp1mUhHazoPG8hc/cYrPqwG6WNZ63qS430qy8ZTcLy2cmj6o9qFb/DF/tlsLyBZNR1d6q1v4AX+2WwvIFyGCpa7ilRu2zNPMPEhOIUUe+gsWdhEItCih6sxzlJyAfLEPmeNqfdAZQ/jCc4RERchtNCptIgzvB9ok4dpTfcvhmyJsCjdyavIeiOBuR1QW34Ay0LwzhPImDdbOitL1q9Uwe0yCu8ojMiQINY4f604MHofb3VLkwHqdXqhzX0gV34aXEok9qn9U6/DTaeo17+ucV6eLQltwTGPFFuorB/MMtdegViqNWPqhU2s+XNpck9HhUKtA21XwHa2WuLWpSjzQA/Lap7oeAeEjUIJJp0tFbJsq2BiebM0BqlTi7ZhGFotymE3waSDAz2LbhT7mtayxIKTf5hSS5ZBiQXmQvb+/WhjxILuPQqD+7Zr4kpWBfcK0dMXzSTNXVL5MQazuIxBMGRvtqVRbobeCBMSxcm22TepIsuqeaHtTbhnIbK3fb2PsulvstOwzRB2GBNgv2qkVgVipwcS5J/dYDzLA2c1uKk1/UQRLomnfVO0vD10GBkp7rQE9NngKgDYYSFsS5cwxllKuLs3DVzxKFEn7NyJLO9Gajy5s8QBFoCpQySrbXF0Ir/yX3woRaNrc28LBTbZVV9paHqOBXywQ1/SqM47Sq+fHck7fr1HFnZY+ksTNyA6x1M1ArhBuxhO3qZ160sLVxgyf7YufQ4XaZNQ7hPYDpQQlebqbTG7nwJ/PesFrHt22LPKjAaOn1CL5Ty5E9bQZcFyaALMxxpyaKQ7XoktsNJLUvxPKraz3rM2zH+Op6QXLj6PUXvDvUtobYqFkEMHDwSD0U5sxh1xKPJA/046587LOxQ4WFsQOcRudN0nJRhoc2I6SIvhDOUiEqJNq5lvoy6vK+YZislrC8L0Ni78LQfoExp2F8CX9YcADX35qSo7UUuj2RzKRfQCheGcI3QSTZL+5lBeJradg7wPPAkeMudcAoIfqTKCuPmeQG2hFwSUQJei00p2BZuTmGCx3NJfhFjZ+xS6GRkXaS0dlmJ9o+D/jzgj4r3enWwQtD6p76lc8KKBtWDZBnQRTXQDJ39Z9vQnmAL5k/5cpVSF+3ZmoVSc3EqllVbs05YQYiZJbgoiLnipJ6gtPtd7BOK23TvLAcga+gsP06oCGQGFXX52Xaon6viy61LDrVUJvCG3hzRTnDq1XGaXlXlf5WSx09MDP7W5F8/zZLvrPuAszx7G9KxHlArdMFeQBSDcAEWq0AdCur1td/30chwBPwVR524IeUI+Yh0X+MtIOfOqCtHCHpwxg9Zm/RvmrdmdeiQKxWa2vPt25tS8BcvNP5zrfLoeXwudgaz8VIFBVRQo7CIb4jgSBmPJ0zL4PMiYPleBpYfw0trvsX1u1aa7c2KG3/IY9fQllk7rdeVIbLwnG/sweQci8wO9WuunnTJMCDP+HEQWeAcggKLwmvHPtAnlbZ8nhsaZaWp9+1sl2GK/b66Fwd1D36VaHXk6yEZ+QOAHSr/gTqTe0YFcMHyC0Lf4zsg7cfzwRSIeqGb9+V3EATBji2jJUQIFIMK7qEMPRAnZdMFupDYugbD0U18L1XgnRD3iCQ5g6eI771RGHXAwk/LxegHgZhb6cqNRQcLltbMbH1aqo7L0/JVO7rNdj4PEGeGz+8o2jgBXkVEQIj/DMAGjzC74uWIngKOEkTWKdjOb2vePe4HL3+r4M3cmSYyNOoyOLgxudjQZC7Zx/f/NOW/OQVi/TKjxJ/hnIZvtOWnp5z7GVPsw0VKPXQbTlPNc5Qu0ZJchcmCGqD/aflEzrAF2qbuLgEpMc3vggLCJ0S2g9LQ1uAUXsiogiGXNkqVsE6ZId8M5yANqaLcI7upnRQwg700iqW+1vKE0A0pTdl0abqEkiyBLvSGpSsQq54SG0aEeL45sEtKJsxDTt0L4WZjvXIa8YDxVBcReXCsWkLpcVnD13AomQVVl0HTN83ZnwfXVziqpibVIvhRGkrFuC8xW3moKqPufYkjjLHgaUNfEf/jWCM61JcEEA9fCQ3kl9dmm2DKuS/UCDAlKRDBlQ7F6BXg3GlDJFYKkkmSxUIEl9+r+2NI1J/qZ1xXU2MF5QGVphcOlPaUzCQuxJui9eZ06Ov0CXHWNVbjx9bW0bxcyrzinHBxRHYSZdhmE2jZcGOQ1Bv4D3npTZZDxcVIXNlPR1ZV16xWrZUfYIUHGxVGOvrvfY9OdCiQwUzLTmhhNf+HFQfSBH0pQRjHSC1fiapDABHSZXY0fSaSZOkyW2Yp84VDt1XeYbnnvcvGv6oEeIJFZvMu56jw7dzjkclkXsh15NX5xHCRBlycnB8uP/mQAuRNxQQJYOgKFhGhE3lFBzE0x09fykESqyP0usb4Zo6SbMb4aRMrz2hiZU3TqOAdF87xif7AbqVqtWbUE4fhcfOeTX1a1QiXc81QbZHGK8lVP1JiO54NtGF3AxlBdIGiOTvR18ODm35lkLE/CK6xW8fgIcHz/HIrunfvIRxCDJAJaLNfllos0eH8mpuaiVCWPjVyccRtKBnrliTGcTRzlivrMEWOmPppqYbyfE2nCAt3h68+fhp/+zANib40svRdwxqMjjo+6YmK86FD+jK1OwK6ULmL3QFJIXX3diU4YTVML5AdVCVC1cuezEawdTLlSFr9/tEe4C9PvdPgCS/nRz4+7+dHdlu63BiBJdhuOomq4eZ7cdyM5unGqpUjjakID4wJobWt/C7aHIWwRzB5RdJNhIHcxQHTJ2KzFG8IfDQiD1CBQzEnwh/rCTFSMGWKDmtl9fCbDFGQTnhacsOva0ruK01bpU6Uk2vPRNXga/sI8wblKcM9ZUU9Mm6Kvc2UfkuSn8jmN9bwgKVoHtx7xEoWUZ0YppWgiy1QWRXDmkbw6AGj9ZTbSNY7WlLCemvO9cEbJhzhiQUqz9liNrMaI4I6nBtg12EW6DujDE/KcJWeUQKeNrxgxxSgcPyelBs50FmHOBQMd/wSEJ4LrcoXWHCVEnVLKoUsGnyVH2aSdCVQnfVROOxkET22/2zfeH3fXBiy0pCbEjBKt9iJxANXyhFJXrgG76ipURBRQoKjvr9/cnRb8f+/z74+P7D2an9XRYmILDizqKEXDqPjw7/+Pj57ODk2P98sH9ycHpmy6Ic7uKvqd++DAPK0Ta19w8P7y43JTvO/rz/6cD+EXUE8DRVNzmVE2majuWCa7QXuHUDM2IZgR0F/LMjwpNpZon5ajkUAf6UA8Cfvk+nablpQO9e5TGh/eN9IKYYzLu0gslS0hO6Qj+GimQG24sCUkZ9bxcDSTOYVaMt9yfNgErDRBy/TMsg3tw6FRrtuPdCSwTdQwo32wwvnVz6HIQwEpFsP7iIroq7xgx1KwKgKhtoos+ZxTCOAZc29014LRkFg3EoeK6FPqo++P+UHF6UBnAK+TYKEvgVhRgcfiOAhEUZ5fxJSjZ3yFkFEsNexvBNyayoUgjyUrhzP0JzexIkKeUIQBevARhU/yncB1laFtGcfMN1WgWPiViJDvcup+jALIPE//n2DJ42hI7Noy1heX+bc8T10GKS6pFpUJEF0BXFKZDZcS6itbV2NwLLGgoTBzK5cUKPIaAxGvZ2qG6IdZX6Fyc10sowg9p4gQA6jWg4wkI9KiSxUkq2b73UBV9iQdQcspjml0vsCNPKqQByO41lVrM2QwAqFIChiel5dOHhnmRkHDJPvXEQkxY2FrcFuefiLKl2ge27az6sqm571Cnd2POQw4IoA+QH7mv4XVz3+tAeYDqLcPfXk6GAFYzUprpoB4xrRzDUeciDCKtNOc7uAywnS3OBScOvF4IlLuqj0YWOYWnCpxmkmcB37n1LHGadrXFbkBfKgPVVZVmszSSJky95ys9JgkyF8YSLbPnJVavpMdSiECWZqsJyZAYKnOCrBEMcyuAWHeDxicskMgNIAutGEYpJm4jfeIoOgbnvm6V380ljzjVYyJilePDTwkJJeV4f9Av0TBi0UQ0I4FPniFhQlyOQMa0IMXgih22CWMF3eQaAuEzY+peWX9LYuLuQk54B3uvzKIwIlPKYYYUDEikSkwLuGIrr1mxcS404WrtysI2ooBfSOhFoiLWqAqeC0NQ+NSg2tlSW7Ybmvji4QNzkmlPuQi+1mYlg/v45lU7z0tn//PHTKZ8/sKdENnYMw/ohB1RyY8dwXkTvDXhrnq+hZs/SIjQ0O5uO3OxVnvKLrvUYfhbaNwMj9kSwCogqzHyETuVfRTaj5QqdNXqvKHqKM6RYtPdKX6vb0Lj/Krsnezdu2hUCYeUG8IlOPRBtGftMDXhl6v+lGhWC/gTK/qv+Ehke907Rh4Zmcp6a2QJqdcS5ClZwqQbuBp+YpPwEwpgEwdShbxjtJ8g4XkXx1Of4fEHOWc6b60h8H0+WizLMwCRVcbCV8zOqqZdNyrymNc0suAx9+JGbaxocWOIorEkrHh9zQ5VS9zW/i1aQnSdlc22ouWRYOU/iwE1fbtH967d9WAh9/nj0mUf3MrzhHRLgLE33mbI8YXU6D/G0ShIE6eDirvA5v7nQzSFNnBnYZvwFd+L5l15uc5w0viTDd+a2n1e0dabWIc1EbcyjK8jgWn2ibrh6QDPI3TzGTlnDhjlhDbMCzV1ESQjdGlI8QcbNwwFzpuDmDIFqM8g32hy1k+UgNpMatRzIbCLQOoirh32VQ+NWaplVHjg4wEw8tsLes1uQsQUrjWZNWknBv5FGLU1IjDfBJQYn+8XcyyAmJ13amsmjNUhxNqmrnzd/vDk8ODV2fu8OQlTr1QpoSulBXaK5hnNITbcHr4BJIZ1j/9B+4FweJgipTiZl3eEehQw6PTDQUwrFxiwNETtAZZF8k0Uq4mKBouytdnnuqgfpsVosTI/Pt9qtuU9lK17Nj1Bcr+IUsVsJFURRF7RRiCB4Z2wBWhLMEVjIo+YaB19py+P/WL1nWyLRkvCHCNcUekdYg+BMtpxF7N2S6zL87dLxIPy6Fm+u6YxwW2BD1fNa9bxevQev2utP06tENH+Nh3ZdidATqxI8/2zLrJA3KuQbK3zpWn90rT9xnAfsRSBc2Qf6pz7t/LDgscKsEOoUG6Zx4YCJmMz9GDMF8M8cf2K6kxH6uHU5jRUX4J9QwG1o/G8VHidvLXR7HrJxop4Brz+6kjo9CRvR/SLbj92aUyw7e5nQ1AsG13ME9STA3K1AzBsQER3pmW2iqJ3DsXYPqdCOTb1y5eWG2t815eMQM0Fh/A4g27Uu8dcl/pqlKSUWexCB2fXYxF86NmP7wSZKVqsZr2S9dnrV29I1Lje11GjHrLNpVNKyNiTkQU0UxXCDx9gczAqk1KaxMUDoNzUYeQsMY4QWUVb4q8yZ3rrmIRB6BVoBhheCJArQgePJn0PMQQlGtbCd/7AwuiuGqals7E2hbjClprfsJa8aLlGxOmULAxjYRdO4Wgatj1MHz3pAVD+2DBuGM//BRBfBIWP20Bc/MdrXKHsu3XtRfzABddgQU48OlMZupRL5leoqMl7nTxVFYKKGrgRurT76WmJ9SXWoxnEIC2qsTopMU+IqiC8rlMAzA91YhQw063oiBv3UCSg2VT80d2D63jPEnkRyFwsh7t6eSxVbP6FzfVTpnczKIcZge1e13as8oZfq/VTh2OwFuagUDpM1cF12PCV32kVzOOujqcfmuWjebVbaqtfaEtV67Ldbq9YcmXyV/OjAPDPJoR+aAzMgj9ka8XdwXNq/3Dsse+ao6AeAdv+YbN8xJNs/MCQ8vwbPXTk8DyLzVbAOH0pn7qzwORxsaYUu9GfN1tirMWRFyekO1HXfH0LK9mQ4haLJA4UGUmHXVZD6u+5DRdQfXRnBdC/dvq6WWYVuj6w+zkgrmAeTBeZrJMds5EoriHJ6etYbWJOAMhnn4WSVhXmQmycwJYdrDBsebtKYdtCJi929YXiemv5blKwYB0llYECWmlRXMehLQp7q1fcUlS0dZth8Vdg8q2KzQoOx5HAQl5DZaaIgBBNhoFBfEdJuAyGAgDHq6EVfL+s2saww2GCn797VRHN9aqD/jNHf3kxswmnltuCtCH0Xnfsb6Nwiui4p7G8P2FX9uqT/KXtuk+QRk3FL29mYFVrOy4qrfpv4WWU/IlsoDuISfbC8rTsnx5SdWn9Mem9VBK/LstR42eO3bWp2t1G18q5X11F1pWoIGdarlRfQ2+6G4xW1uPF+Jb/IHSX61VoHvvSML/frAzV5JQasCrZ+QBX80ZU5eAQ5hELYbUeiXZL+YejqhgCujnll69M+oezlAOIZsCWammLfU2S1rBaWCYq71jaURnOstRTlP4dCWzCcYBm0lTnl3OhQaOd5l/TahlIlFwJIKMTbCr2mrOoCEjFzW6lj3PE7Q6S6bHaLfg7cRuQj7488eHvkS+vKnJfjNdMVFlzLbIQSFwQGej48YNItyPGFVkcU0DRE0KyHKXxpKGOeuHKd+YXGFlXhOy+9RNUs0jFTXP97Zbp/+96pHSJSIsVs0b4vmp0nMthVsWiiY2ppxbnMmK/hdVqPgxH1KR6pCkCEKJFsu6wbfSkZfmA5VKcKhavpqSKi13h2bf/I5PrSlQHL906u1gXbhqGVaz3GbNiG5pYWAk2pxmF6sp5Mu/DFWIhtqEf9GrYt4AZqbHDZ9v3fKTD2HigwRMxUOVRzBtQ/JWzAlTPHiS1lsk69Z/yjcUXHR6cH2v/w3rO7xuZpSyBWQ4L8qAeQOJvVp44cDYOHMrTbaxy/GoXwUFWf1HJBKdSgIUKmcVS5NA8pkJJUr7EP8WxLeWji4WXCCQdrZeW5phxsup1AP4jy7C+ZYBpgkB25QlQMHoYXjMxN7QefiT0ycm7vPMFUitE8Qe/8wMqDROcdCyiVX3M3PqJs4x4lTqQT9Y2N+pxxse2o4p46I92EjFh4wBa/GITk759ki5z3nFp9OvXpeYY5/CiSzQjwF7fVyNz67P0WAmEw9qGM8mpWfcuJVzjOFN6ESTop0VtXumJjms7TdIzXxtwA3+KBF923E2DyfpX6Ms9X86DAFAAF33WCvu1lEMfiGJHSYSbkJ7bGNANh4lmnIRCQ7yeBT+8PXz999/rLU7o4RR1jT1PKZ91Xxz9L6R1SDZCosEJjYrSmyWw9vSP/BSNvpt6wKygdFnC+k+gMm0nJiTXvyaUpMwNV3H8LE8txoXJospOofeFFhY8upHXHrAbOMS5/DACYiMJ6k8ZpDkAQQtGpRY5BDcwlfwl87KG3HZGG0ce43bODL/7HT/vvD+y2YjBB5mH7DGqgxj65rRC0r1jXQvsIinoYdWLiegWqbEFuMve0k12bOeO52mPMbVxNFL9trN+xvSy6BolWyRWfXbtVqNm1yhS/IEeqHV1gMSd36Gu8FYsSwVv/0MFx56D6f/2VVkbPn5FiHuxc3IXRI4o4xIuzUlijrUOOVMJkeMssFqEM+SpSt2J83K8h4uCfJ4gBBqTikzhdqr/s0du6UdGsOWitCWYFLEl1Ltc5kslpKQbrqCZEkYZI175prd1vq91v1EbW3zM2UhLmgynH6J33ALfHuK7vQTP0Az6iwoWZdRkiXPfCiMzEgLpaOGAiQgG36hF2HfMGhJGE6ygmuWukoYpmGairMjOZXYuWc9NbhSYNn2BzMBGa1MKz2aY2F5g7PFsE7LssAuP88Wo2C3PTn1kAx1i0NC+yYBKqCwVU4BrM7R6LkEqV2oRBGwj64uWw9osdtwo+wxEwEgLR0Ah5yQfxp5htO/8ML87C64/YP9ssLyTNCKGZrw1vB6cuWuQ3PN7d+O0GWGt7y5gAyfIO3Fgefwoy26ygpO2p0L8tGY+lajarmcgbffEwX+R2HzObGS9vdIsrusChIUkFEnxA1SbuoV5VdkI3qRB1E74qgEU6uQxLQqAO1ui4WRtKwuhzUaFvuiZpxLsNtaGgqi11XrdNERoheqgtlTOmTMGIvmsdnXxC3CT0SiqNiqHI5jlaRoUvLxGi5N/aoDKh6KZFDnWoZ1xXpOODahAfFDu3KXJLAhchcX/bcBRGVt45+u1M5jnHO1nw0ovCdIfkgs5llExlXhZlV1aoODMdtgjMOVa6MO4LGYMRgsLEIRhVMgqE6Baw9vJ/i37fEJfvZkShavng5OToRBIkIQS4t+IWONxhwvEFMTAJkiJlFSwJiKfEmAYU88CDBoyWGVgGuB7ZtKBVSze5dUXLKvX77lWVmZJD+9719bIw9hkxX8QgOZvvVkKDufW2pE2Yq4QPbZU2XbMk/X6rS+rWC5nGX6EbmcEvtPAFBoVFMF8+6ov7FOk6GdlpHx5kN9sIwM6RGIlPiVnVtS5ZVwyiz1e3jGxYbsDg47KAu6hu7ujUN5ypGsWCScUq3t2sMnHZh3gBC6pJmubT4i5IrLGLak21jhnZB1+Oj07O7PuH5/JK7qpLHOXatQYcoyXrRkvDTU5uccg6+oKex48vrxQsvT4WgVNvzj4efT61uw8Kh7CMUUCrgwzbZF7DTcv2uofeGRhUtauyWiPK2juBTCREnA18JBkM2ADZCqQdFMj82fj6TrbEm3LxRiHFknh/0H0s2YoYVqxIUWZVBNjKmaSk4jCYcRYcwY1GvsbLkPo0qrFMV39Ch2YUC2Lbrs6o1XJJHPh8X5hkfP1dDCAIxRKErn8Z3shx7Bp7IOVCsMmbo+M/MIPnchxOfd53CCWXyjRGZNFwIPrn3zAeTwsiMMCxxasgn47s3p822+Q+zD8bwWKCAxGq6DOAdxjoYmCSw0jnN0Q3n8x5qleEsJCc1j58sU0+wZHuWjwkNDLizg+604FVQYB5xI27gcVdv3ybb0VNZD5febSZuaR2oUuvFIfBw8/I9k2Ct+UqLdFdgwnhrRYgBB1GToBrFW8ts1xLNR7pBkPcAYIErq88d9WahrlAjNXng/f7Zx9/P/D/vBPWKlODbo4t0rhrcU+pwzT/izK+c4jUcG5PPSirRwke/t2j1BqnC0MGTbUMWYHJpX52yO6g+TxOx3grDd1oNej31fH4PbcY1oQezlLEvIHy/Uq4+V8rTnpokUBLdG7hFqghmrnHdMnykK6ZpguXjSC2aUrboJiBVIazPcWkY2D4YSHcO0XH7E06OfOz23bO4ZudpSEjj5h8fm2evzGAasYZfn9XyhlRQuWcIbB2LXpjY+KZ2woQ7SpvZpwRH1WAlwoXUmG4AkO9q6Myktbjb6FkJbDvwljdLXkjBZMEFY2bH40NlSipbKc8MHq0LSaQcBVe3LIE7vTcjMQZ1vXQut4Y8gkYy2BhXPvMGYBBmQ0ZfMjCoSwHuMOFC5Gl2jyA9852Iy60lrdp3pa4qZLuTzdM2xnEaz7FqvFVjo49tN3z3uCimd4LMBBIASK0NfVsVxwxV9vzgFdmnHoOU2yMLOfxpJGvy9h7NyOpmvvqyZT31aetV1fR1vX0ji32GtT23fD6Dk0rwuaoiUXlsjogmJxH8JvM01PLoUWsT+xxYSSJgbKKQas5r+iTugCM45w5/JYuAgGNscYZgIy5Bl5KV0lp5glXGaFu70pNtCEno1mYhbRWXmwnjuzTg0N9xtlSAQUmtM/WFyZWOvm4f2g/FJnKCSqbjpwt5J77MynfHRmhInBWyD63GlnO/g5YqiEV0o3nSPp+YyFY60XP02qmJtxIq42gbkAWEumELzwVMqrbqeZ2qMuLFpvivv6lP7Z8JwhmMrbNyylDhWe37WvrBywI29Y4tbV1I6srD7DMMEkoP4wYP3ANaEfbsI5ssSv1dZcUvavvMOJDVs4XxMkh2ZzAs85iZZFsxDXK7SSmu2jwlkBxU1QhMl48st7CQoQi51mUAo/MVvgGgaR4BIshwFRGp88I4c04DjxpfRQbrA/ErGl84NuK7XG1wfa4utf2uNK2B6UQeqjpcVWBcYfpcXWn6XFVNT02Wx5XmyyPB+pi6EhR1Y8knQccEM8q2aUrNzdozbXInjlbwRxuMXC2Xfby/xHTpiTXBExHppIbNHOP1MUeVqJEJXstCRYnpk2DgLHfKhsCdO2OhmC9C7C1YoKlAVpXIR82+YFcKtRTTr07PNo/898cHR6d4OPxkU5WRAM0HwfmIVuTdHcfuGF9TDrJ523rSQVBZmrzTMsmbAERrNc83WrrDktRZQnhl4fZgXdabSIZ7gNsqX+/HTUJqq4Q7adjv9OMYvvKrMu5t/wNgsEEq0+EoFbzPGmDKed2Hmiv1Wf9QxUr1Psx7Xll7Mw2N6B9XFc8dA96sxb82X1bQQJiT961/f2gGQG9YX/1gW3weXIN4Y2J0q7u07tCNLCmIq1b0BL+8OjtENfmBd/LRpcGRvMQnY6cZVQsU70VZ92Y+T94Yw4gfF2FcwR5FY6hxCSMMRMssDbv48Tp1KFEoJXzrzhtqsc4vUs1xspynNmAs/8NrXnOMPoYs2G53x+SGDZOa1nf1Isl5jQ6X2qvKGhQ6VByg9KzXGVZuag6UgGQoXFCZuhgMUxiIlE+1cr3O5KpGolU6a98KVaKcZAVOPUjih+HLpmi7Gfyqd55hrXcasX8jhQ0olo1D80GQRGn/4PHam1iBf75Hzna2nRA1fDC3NTrnzlj2niO9YOnTZsOre4KXLrvKInPbBJghZp3AYxtKexHTHMx9Tl7b9f69t09bz/tlquZOL0rzXCbFI3bVi8dP5kJiM374unWcYxc98tgzpeuUPZGRHXg7/bpihVOJICvtvzBrrpVBcYXQT+G+tarkTXYNVMu+1n7vvlMbpx/wxa/i4VIdYQc/KTcxvMR4pcBewPMoRDI9MJFdcAbub/jJeoowYdWMM5DvlTdW5TL2HLoqmnMPgGrLWwPPVdD1BHkQiwVghD3VFEuhKj+CJ1oX/7HNJ2Q/MR3r17SlxhE1sgOCxuew2D66uUyBPHP0Ycw8VblrPfcftXh15SJ1V5H4RV20SanRVjDjOyraAp9m4ZrMF179NCNkqiE2dUT+9kIo4zKOHzF3dw/trbfvnzKrzovi0kegbbmzRag7ioObfRnHNmLssyK4dOnk2nifS2A7NE6B4FXPk2y5dN/zNN0HodPaTx6iFiY/2Pb2/X6TzEDbuU9JqQHCNDRp9waNlveQPN4d9a3ZZDPo2TYfzGDTvVmwTKCaVDcFGW47K2iF+Ngcjmn61KGjwZ7g+eD6QsyDoaPwjD83jEb+kYEGII+XF+9WNCyYvh8a72owNja2trdmnzvwLzJv4krKoazOLx+MQ+y4fPs+gU+9K5yeMJ/XmTBdBolcwCbXX/vgIVepsm3CsTZ3ixUWM1msxfjNAeTFrrEP3oY87gqCLiEBr+twQ62hr1GZ9XhYFc34KWzWaWRnZ2d70A/ItvLp8wxSD7kEIMEVjSFYVyLIWzxfgBzGvd9e8hBOa5Bg1WZIg0sDGctwBZPr3oRclcRlTcje2DTvCpAv8GDh4/JOsrTBK+s6JFX3chOwhVeQo8jbCIDuE2jNd7sWICsH5P3P6BH0RBYFj6+ai0Bs1QVkCz6qgMoF6W1XI9gNqHXd4nJ0A7iEH++vvk4dX5Zrn9xu/ubv1PLUORwcxFs+hf3hWgMn0a+DybXqe+/6MxWCUcbjMvEybpld+Z+44JjDZGTeAmgzi88ngBy7OFh9Rsxc0t4TpNJHE0uR7MXmVhsvFlE8dQZuy+EP9oYGHyNqYsO1lDpEGYWqI4csQymv3Qdd/Tq2z4saeDdh7NPh6NffnnhYPk1hsCN43Bfab///u/zCxfXpQewLnWS0asO9WG/m4D4h+GfhM4v/i/dXyygDoFFMLL2Z7qI4QW8QkZx3O8u/O9FB4mjQDrns255ATUR7CGQRkJBRpxBjRcdNf1fPiXOBTZGcSjd8xEcWqHnbWzbxQiSOJoGU5B9ywwM8sC+0AboZdcqWQEp3YOBbWjKp9Zu/z9JEykdBJ8+rW7Y0k9B9fxn5a4v0EyXnDgzru+PUI/FsvBc+KrhvZtiBYn7xlaaoa5sbp1ppYLI2Fe2iy5os4XhCbfwrvIIzHIspIbFluwHaH8tQDJMV8uscBAV1zXPKakBH7QYLnTooWvdgwdpv/18sojW5AguA2dwp7BM8br0ZRbA+gf96zqZP8bPdx1oe1RCH2PTo9Pm9SdNIiQiefcahvAPpHvG8+tgHfpBAVZllCA0c51M7XeJY2BNJk1AXFcp++bv+kOiMRdYT8FmmIYp9TonGrJpp5Mx0svoNniBeUxn7PMRQ9npCt1iyQ8U3fEx9TVJ3zKg5JtQBhgV15gY1/Lm+DcVzVK9QEOIC0EWRsEzMXmIEyS5NXIlQjFFn1S8H7X3SqGVEloNvA0zTKxBbyYwPURa+iTFFHbzyp0D/9/wZzpuQphNKQpfg0I/gqCKsFtlvvggXbmoAVEnTMDa4QY4d5xYD64rbVaoA8w7K8TWHll+16WH70IAOcH9wukUz4xs7oJ9YX6s7P6yjLSPjs8+fiGR+tvb/fqlTo1dIYWBh/NoVeKAIo3kvV7lhtKgWUXJwnE3pbrWu4e8AzzlCzCwvqjbkuJQdqdZeLhxMTdFVsQ1r26wtSwPRPs3WBsEl9Ukindm+2uEFz1Asm1gxvfAGEgzRO8/BKOIGycUz1Sqkj8kbe2Sj7Z47Nw7Qdtve3lkHX18+9lyVgXl4QNscBrpOMutJ+4L6wj69EXKNpib8XyVwATFwlPaazVu1qp1E9un7KjAmwefKV7s7cHno4+nB/ZdVTDBG3Zgnq1M1B9MZjWNizRe0aL+WhKM5DnM47uL3zygOOijpV/iHR/q4ElLQS6J5rKOm6EnqkDXeeCW2/wLu3vzj8S42zryoGVgMdyP0mnY28COYz9PYPWeg4WSlxHYLna1JRAc6BXiUyj/heCud4B9NLGNeGTuFh2bAjeqs0uV6+zKPIS4SvN4KvbJqIhtuqLTVwyv54rtJxJjZMurxtkBBtjRha28QhJwx3O5w99v8dDA/EC7HA/PIfy7ymNDVxw0Kw5EVNgj63A1oQTK1raFieBEwkk6I6FUsXE6AZWDyYLx0iu2SN/EYKJgSP0Wbo70trsWLJTBBvwVQ3S28Oe9W1AYZh/DbEMHYwch9LagKp7y7fbpqO8BEHDlkQc2p6PYRlTwiHCPkDDt30Nz/GJcROu0xrTDebBvKM9DD1ckFHgXYuyZQ9d8ec/5ijqjmLgFrBKWmW66VJTbOjSi3Fqy+MIb12zRKKyyzoarmOSKU7118wNHyVGmLg3axdTUQCOwRjFbrcPuwejpiR8IlOM+8EBapbg/XYFh3Gm4uaAbcUTHfGDmo8PwdOogbUaIGuU9EEiNZJ4geUMybio1tnbFJ3V4Tq3yrL3rUJCLMeANB4L/3sPABztTweykeNeBTFc2aAA4SVfzRQJ2fUt9vGRb0qTW8eqx3iPrTbDEoxw+alz6U5NcvFkic/1SQdERKooXvyJ5dvvy3cZbckXlLld8yBW5ULBSjFDBqwGCpRmyho9segbREitR7nO+E7RryQyW8L5lBjkqXRalS38sKvXo9SQtqq/lJeoI60cmmNn2w6aYTlbBepVPch3jCA2bNLVsy5KCQz61/luG0yhQl9H+/vHtwZEtrRtpyex61w9OxlCHj+tOcW6CDbx79+n44L24CtGbzZZZOPf0Z/y4U/s6gXlE90592NprfqMU4KWPvnE+35VERT++/1BvhP74uOSlm1Ps90dHb1UZsTomH61yAaQmEhfGXX+KjMzP1RVN5441mAlJmzibQuvE0pkiIBG2ilEUt6OSIPT20EdaAxb5OMxzIXpnpCCpcEHCHeDIOW2pGadzctJUktNtY9om5nrm5L0ttz2FPToMVhLDmV95u8RJRIKigbiqOE5TdG/pYVi7yKJA72cBdJzMru1K4MuGsxCeG990rwDwFY7Rd2+Z7ZgL3NaJ1LpsVica8lplqtdt9OoJhtDbE8xwU/EqEWyitg5rJ/i8ZyYw123dE+z5k7Gb1IrukeKvFoaCjqIwLXHTdPjjh5E6NlNkQyJjXf1+SGwmX1Yl3M0GW/0NjKETNmbReqOyOY7WaYkmZzVTyD0udNHadLpZbvCTUxOmv3mGVGoDXBPpFtUhDZxKsc03CVA9ley/Vm0T9C2Za9JMB/kj7Zi59DfeH4AAW9P+uM0bmHiTeaIRyGBuly1bGJcYoxDmGSyghfq2Dz/iNYb2T4skwWk9Ef1+p6ARs3Ue5en2Xv/nRQsCsADCT0oLrPn/Uk4AfCUh6Ei4DHBvaR0F1vHn9xWh8bckxE9Hb2+I4Fa0bLOIWjf07rGMOFvPj15o8RCTCOho7Kiapkj7OUUZ+DwCXpbMm2xDhyXA7lEcb2IcDYLvb20A/Xt8o3lEsY48jeGTkK4lzyDEYQSUEOD41IXdzwlf8YFcuuWdabS14ZTRMvTwHweNhjNOJXnnYVMezUGezVIPT4wq503cLXWQ5GCjIOK6FnrpJiXmpoE1DYbQBsUkiqQvj1hfjYoydzs0qzENXVo9k6JuALf+XwxEIwQ=", "app.py": "eNq9PNtu48hy7/6KDgcHomYlWpIv45FXc+CZ8cwOsLPr2LOLIFqDaJEtiWOSze0mfYUfk6cEAfISIEiQnKe8BshD8nz2T84PJJ+Qqu7mVZRs7+XMYmWx2ayurntVF2VZ1oc4ZWJOb8kVmxH7vaB+wLvEZyFhMs3gYgwXcSq4Got5ymacXxBOPB5xktBFENOtJJuFgUdhLMb/U0EliymhBL5dMgmPkiwmaRYDCPj+JuSZPw+pYI5lWVtBlHCRkkXIZ/n3z5LH+Xcu82+Cxj6PiiuWf5PMEywtpslslgjuMVmMpEvBYGPxohgIIrZVLKw2TaiEb1tbc8Ej4hBzD3YzDxZmkMWwW5bfWrDU1SNbW0fffPh4RiZkar2hEVBEWD1iveFCMPXtjIaZT/Ovqf72mgah/nbKEi65db51cnp8dvwJAd1tEfj3DAgezQQbEzuhksse4V4qGOsRjwKNeyRl12kmaFfNtk5pEvjcgtmjvR4Z7e33yO4A/sH3we5Bt6dnHQECOGcXxncOdnvkYHXOR3odRGrWHtzZG456ZLinp+0OXu7DtPut4786+njy9fGZ+/bDKWBsbSPnQVS2v8rim4zGO2/7o20qJXBmm13TKAmZC0AXTFpbW1s+mxN3HoRM2n53rFblwmdCETHiIGzcpTHM97mzCGdIJjMqgoUzn12XI84sZLFfmaGpEtwWzyrwxb98Ggg4TQOPO3z2ufJ0MRylYWUYuC6YDHi84zsS7zSAJgG7pTLHNWHArvyCCi8IQ5pfwgbcIJ5zB6UcuI6PzyVuHB5JaLp0PvMgtv0emXfJnAsyJ0FsqBPMST6JXQcylfbqM90C5BcAc16CkCC2zLdR0Rz8WHnWem51u93qxmC9OWo9Pg4AQQHV1Rx0wZdXQbq0bcsJ+cLq5cuCJmYCJ+dMllkUUXFj45YNpylsVlEAdMi26OdMpiAVPXJ3rxcPQakUQQpU5tbz53fFIx0l/p0e6fy+0x337rU6PH9O/vjfuUqQO6qnwnWne0+Sa7xpVQAy+WPGQpZyUgEMvMHZdmVomSEvYbEB3NAXPaL4S8AO1EBWnkrMQ50+wOvi2pWbki0yMGXSTXlKQ7OPeyIrQjW33nER0TG560RZmAaXwGzaQX6UYCo3uoSFkpHOELid8s59DS27bek5glerKuw+GbJ9OBrXt9SRQWNZQ2E3KJeNccnWZczkcqEq8FOQIibGNQaooZxwhiBasqpIWMo2eKCOIDhGqgrJcWiSABjbOionwa7IFwTVT0s7QprWoJwbya8tE3LfVcK1fpGvv30LIwsm+MoiNRMBtLxAIl0WAtu1lG5e9Mgl6pdGqVzx3AlSFkm724ZYRMEjuIKFAbtk65E71RPGBOej742R7WB1iYcuEb26hNGQ5npjFWtZl0yAHWTSlUHsojzDcglgBmy1cnzXrDu3TlD+x2Q4GJDfEb0CrAufMlhkjFCML3L9s+qmtCm4084GTDrn9yS/TYrRrtkGkvcqx9UIDigMR2MzPe+uxf5P//zv//s//0Duru4NIGPVrB/iH2LDXPVIF6zc6Yf3ymVbR1mK3tei+BeE4KssorEaWapvgc9UXCACqkgNN7yM+iJLmK8eOANMC7Lg7RhilgxA3BtjehmwKybcZRqFtj/5BuS2R5YsWCzTyf7uwOwHYqpjHbuRnbfExuCHga8BXY0xDAvmgkYMgiZxGVQFAIQuYSlVcR54OrAt6D8xQDMCgYbfBz+kvjzghSwTPTqIKjiVktKGkkDkL/3gksj0JmSTjt7E+E7/vU+uD/1AJiG9Gc9Ddn1IQcHivlKHsccwZD1cFRoL/Ugwv+mbQCSfiRD6fiCYl4Kmjz0eZlF8uKDJeDiAhWbUu1gIDuZq/Gw4Gu4MD9pgz5T/7WOomMnxcBceBEBcjJ+9pC99Sg/nsGh/DtFfeDOWN+DQon4WdF61gKruWz0lg1s23t1Prjuv/u/f/uPvv9yGCa9w1quzbIahM1VWndyQJAslJW0wZzlEgxU7YC8Z67x6z2ImqEBJwOj8wxEYqONcxL7cnr0yq6lPI+xSeCDPc2tbR8YuhJXbGKpN7pps7lTZDPb69+kEVDa1Mb528AOM17211eB7LoHCm3Tu4PO+kyN/FfjpEtgy+N3hqkRoFowHDfa0sWaVpx0wOSG/AopnYSjB+LH4kFyLvoT9BDTsQ6biXYC2dRrA9WPlQ0ApjX6Ds3VxRmPaV0I7Foj+IURAkCn0U56Md5HPX1ICajkv9g85AVinScedhTS+6DRYbNV5ezA7mM3nh6XsDHcQ5tFMBAKVPKFxqmwt5GgQdYNO/+lv/+nLbVpy2YRmPEuTDLQXOIkGsgdGgeXReAIiENJo5oPwQR7QEpo+KhZtM+0YReZTYde4hbZHySvIScB/qBADbZ0GBlYLUMPrwsRXo1sAFAIyoO+QWFTsDkayELhTkeq41QI4jLtWV6FjLQLBd/YHVhHsquHEnldAlKuvYqsm3SLRfAwBnNsg0TxUOQ4M29Pb5LyFZreJCaLAH8GTRU5UU5qa3e/2EAuIQfM9GNydKNm14KaC0GsE3oa3hvMLZRVSZkNiG6c9gvrSIyGbw3clsDB9yVj+x1UaBrdYxC+ZO1vAeMqSMh+tc3lOPaaTU71qT2Xurv66yAKfxh5TC+XeCy4gEk6Z2hnjDXh6lwJhSpVHInCgS0oh88uTYASogx2Xx8X3BqRE8AXAkZOFcE7MdzuXEpQQJAcJpBIvJQFq//lIxYvRAFgGUI6F4MK2lKGGoCZiMeQFoTHY4FjnAp0Q6I8y40v+WUVhKmiXXaeMtgoKlWvgFQiOrno4+AdNK2bqz5/vDEmfDPXTLF7AtLIcYXdbtzMuyfAMlCXkDWyKdRPUEYAI0BzwwUHqKiLYqxJRardeaaKf1YGWGjIbTMVNNQLIoYss/tkS2B4zNuQyF0Xlk9h1N5fHUggxeFP4KmmEj+46yO2S+ljhXAPUiOlkHnIKOOgrZVpLYdbmYeAM1gEppDq31z0SjYtRGy4hyPMmkbHG7NpjSUqO1R+IiLDyVRHtvEQGDpEhV4obxYiTCCCoC3Ds7jqNgPwVPSfkdqwRQxdeB5CbWmCnrXPkK3xHQ4UXyC1lrXJzNaOpt7SNXVOMMsR/SJ9VrQIfe0BzLynE5FIprbQ2ahXYdcAh5Hhzeo5ZROGFgp52RCzOImNdcemKAylYEpBtkPXYTDD8wZRfVSPADQzvt+/KCTkFtXamS4zOVDoI+RCoLggyMlmmwriSuRNDhFJh3E3diYmK/iE8RdRJiypoOk/UZ92VIx3yfElMLXR35/UZQKNKQvUvf0OK6HGmysMQHOJV9x6ik39suSmmHYDaOa/t/gHZXV33X/9u7botsqm5W8vy+CKXwujGVbVAkLAi0XoPuTSIDlpRXSeUYFuo8DEJRbcN3AGus0uYwhs+oVvmVpjXKHkqZMl/uF6nS9POt999wgqsKd6hJkE6LNnkk8hYPfyppm2B1AFSnXSYOAVxxiq2AOU8Bhtm25uKkxiJwFZdvVUniReYzuoxZeDVUGsw+LgwEjZWRoB5MSRa1NFXdMx5b8PtHllhvd+djod7545gkF2C3lguYkqKuqcRBAVqOt4fnBvmcwCq2W+zy3SMBuQMcigvfUtTasgIJFVs9H9NFm6toVXOQRWl0PjGfpiIBb/W+JHVf4/kqyn9YrSCNJgChZwg9tn1+VaL7D25ZJCf6yyzNAjL8EhdOh5Pbuqw8gtARbHcVTG163ZX1gDDa2Il8DxYwrlfjVbMLazVO+CkIcEASVhFvl7U765xtRVXQKVs9YpFLramQK6H0D3C9XC0s1t4ySWQlUO8byh3i7L4VPlTeUu7DOYmEtRir1AL8GY8ExBj5avWaacZl8ga4/IAgwtvWQxeRhg9qonOZSDSjIZuBMEe7qeYhMZIPeZ4mU8duO+ig0ZK2SpoqtwMpEsv8YgNwjDb+EUMngcVZ9Lm4Ou1ijtDInTQIpgxmzlJ9x79w+nRR3Agl5GTSQjTt1VQPhg7w/k9XKxWaAAUzFX1/nLyACa/f4014chJmMB61f3vVJX8/cl3JJPUp+N2WPZiOjyHFGAxHUDa2Fj9Tt1srlKrvZegilKfxNKBR70lZAoq9mCOuir8b6vvvV6p7EEYYkOoDQLy5gzroujlZty/6RFH15NUfY4CxQW5qxVqXgwPhj75Cy0fFGix1fJERK/7pky0Nxgk15X5h6RW/Lrferb0BVkOyV1ZKhmNkutDossx44GZkpC7ejEvnwCTyYDgNAkhUAPd2dAf7R4SU5YawlTIqgKfPBvRkb+zn98pClP7uHJCfTyIVqVHBJvSxW8CF9F1ZiHHrMqsoi/ri40Go9FoVqOhgW4IotesMeXZAgSlCgUr4VT0Fa9AhO2XA58tes/2Znuz+aD37IW3688hWmhh1Hw+rw4bUaoUtl40GNzY+mjl9nVfLkGYr8bANjLahfsGj709wBystGQhca4EBZ5jCRiId0/ycTAW8HlXX2RUF7L7rTnnKUpiXqGOsVZQnYASbxKXLAh9Nwty67gmqTgx8XiiTeiSRco0CUd9lc5riF5sSLqwnOMuMzaBVMkPFnhmELMM8rJQj8oQsg6rt8m9I20n0xL0e84XIXsHo7buBUGvaBUFbHT3ksayLyHMnUOI70hwP8UCqNduKQnoaMOJZTQZnm277/pUXFQmlcBQPFugaRlFaG0TcnD5rCY4xUtXyRvOUuJcwqrcLQCZKfVNYh5fQNH2Jd9feSsHkd9v4KLEqw5J25sCnzRIIVJpXasSkECcsInorRPWUz1LUx67koGN9VG+WhjgjeY7Csnm3M2YPgS6oPgj4K8lLXgo7wL03lC3Bfuc5ZtnrmV/47F2QVg7qYG31vcZhKHoGU2bDUYuJ1PrUlCIZ2bWOXk1ITsHJmTR/TptU0amNJ73/WjYWNxG0/EaBUraSqIm1tEJ+YjJC54DYZUBUFYGYKI+AX8pJ+CuW03HZzmx7O7k1Z3PvSwCA++g2DteCCHs1xBzOuB67A7uE0/suxgW+BC5lYFBjtIpv7IhZ4rcwJ9Y4HetRvoJUz4CFLDeMYRez0gT6R/ilXDspFYdqC30I4aQ5nD0HQVCNVZ7RvpP/IcHK4BNcPtjFjDh8xq4fO036njRlh4Fuu/0SBTEropXJjsjCD+L/aOHbhKgCucTnRVB9YY5tmVO+Kw1c6vVWXjkA3Zg2UpKJ7reVBQkUCZuEhCWJAg3O5FGAo8gVW48sU7fvz6yijPp0c6guxZOvokjzwMrDPGkbX386Q/SVKOJzWRCQ59ir40wpT8sWHcBPKZgrTxdtwYKxANTlV6CSWgh07HGpE6fDdsejjZsuyhTsXkbS07UZlHGnF93PVVUX78guMFfcUF9arK62Fe1E4fGek8C7ubdgrDEW8ETZTQeJMJUn0r01DmDOnJQZMEoR9/QRw/qDs55jBqYJ/GZEk6xAA6piyZ86/xh2Jc0xLBuHc6GqN8CJWLTY4MaYj1J54pmAGKfBotCuxrlwzUSpel/ijmajaeuNtaQsWysET/SjTByya+0Y9RK+wTbQkqDqWP0p20OfCc2tQlGwydsTAui3tsZZF5M2ANn1CN7PTJ0Xuizz8nAGQwLFkQsFcXJwWONHXZwIdvyJq4n2LXK4ZDC8o0JP2zcXYGVaf5S21eV6DXNXg+s0iAE7HrkYCHF2S9IUVICc4gYfBv1n0aNN4AiPvQEImA/bm5lKiJomqgLKcRgq0DvBGihqmkPbB0I1LAtU2s4GO2iOmOjNP7FTmg8rtLLYNkDIjR4ELKkJ0k4OgONXd6EaSfX3QcQfKJnK85NN8nLX2ZBSgWmiEU0+QCN8GhzE8R8Q7r16Im6q3szf/qvmNzoc6inCId6oMFC1aVfGKeyU98gW1kQg0695ANikh/0biLC9woVbNIg2GGyHuIsNer8WuVBRWjX3rxlVYLJBYvRzga3sK9wsWEJJdabi8LVvn5VEtYl/u50fHC+Fi6kJ9VSYNs/PHLVrwFIGztDVJoqJ1Pl3c4Lah1/ZjBnrTGthb4fA6kzBC43hb8Ld4WyRx5WmINbxX9NNxltoNuChhrAexqGTNzkAc0n7lGSxfgmDIg5xQ6skGMhtXyB5imhtO5NlJNREQ3tYdKgetD0IQy7MuK/FugSjxI0su8gbFZHyJCLZnEKfgo7xoEDpdD/9YeTypHlY4h+dEljfLViE8WfYJ1Uw0bNxQxhx0N81QTMqbqLRlY5mtLNnODrMA8op+4BqYN+CZTd39Ww9f0C+M6ojKj0nfXgVVtJA2nzdoz+C+DVnAL6nho28N9gfzeJfvoDpv+PCRseQ0fdLQSTv8miGWI02tktVjxjEb6CYmHLBPMCfI9l8kCYXXQkNWybDuEKoxkymnIRbIol8m6bOsHwlSKIqTCUMFHEXgH1HUxR8QCheTPVBvi6I6cGvT88QCnCjxz+y5L+7wO0qjKA5By1NhWYd9p//M/ur8OKoiWoGULsIj4vRkq+FX67u4MigMDyTv7gY5KDDXHEKZNgSLQre4wbM41LNQrumjfIUAn17UKWdyuCrLmk39p4mn//mmNfXLXz5kmp/czNOysfMHRqGPNM8DWWymmt0tu8q3T8rF/p0ZlOUTDIffRDkH9RmPIg9BW3d6LeiwTZD/mDIfDM1V1O1arcQ0881fNsUupV7L+HhD+TqmJ1evSRbOMp6ePctygxW7OXn1ENZNJjwJngEXXAF9XCHx7GtTlPdO4ax68+ffy61vPb0keNbylmsrEnq4UGLaqnIlL5GG17pNW7VO9qamQUcHtVWMuq4MEGz3PpQozcAuh9Hj5vBgNsXrELBsJb4BcVCywtguipgDrlPoeQmgpvGVxySSRorbZDj7K+rfLd3SqexRNAmiThjatTVRvbURpUxF7VHnZsonia1HWK8+qxdn62ns/GbDOtLAX7Vq1HKDQ9w46eJmavJErPSI1ugz3fqmfSjrek8YLZVZR7JP87be+shSSw0vcHKut4YeBd2HmHeRslp79J1/lv03v+izvQWz05squHIuhlAnynd+OGQRSkGANccXHBhHK21ft5lldphnFLauvG3zGxKz2KvbIhyHTPARMhj+npDKHCNRh0pGpms8s2t/wRRLXC4MqiqisXYM5y4VKOspd7tXN1Cx7vaVfy4IZEBXbRV5SjoSx4t6JaEdftWD9/6woGOOqM2aCpFBTYfYgjEb12lbvZ3601RSGoLfj3dC9S8yghZlZiyz07Pv3++PRMd6Tl78qDxLvYYmDjR9mB+iZgAoJYyDphK5hzUuKx0KcKGCc0TsEY3pBZJlWmSpKMiZSTMJiZn1+odthx74KlRRPqhW7tDGJs8MCK1xWbVR0XZgY5rk7CE/uipUvTx/cPsP2v/gJFa1+yAgoSwCVruOf1jXQrDXVpEysrzeKYYQG4RA57xdoRS52UCQy50hwHJEWCpBDKPCKlekTR6wtSvHRZOEtNQ0f/sdVpqHzEtqUzC7Br1Ro46j/M1Fq8vpG3pI04356plvZV2LW23hwC4G9EK6RZ7C1dzWa1vcmLg/18Y2qjkxX5KzQI7pVdLqVeaaA2vmUKQQw6tElla2ZYraXpqVef6MKZXFLB1p0VYBmExamrf9TDxWPu4jF+5TKkgRnIT5SwmrACRxVVmO9i5QtSg2YvZLUMZgxGLlXTXCXOseEVVX/VFCid9YofOvGLDqBrVvuxjMoUa0OzLDxWEbTyF05UCz+YqAVTh1j9H9Xnt1gbvN5wzmIt0zSR4+3tBYhsNnM8HlUwqSK1DW6MUcnkNvYWyXQb40w0vNvWeuiV5/tAqOy6TyN/f1f5NCwjNDqbYa/eEmw37hLydf5ib69mXWG4Lqpgh/AHUSRYZvAisNkMBGpi0cTSionyO6y/BZJPr3fg4ggWRPUPyDgpvwD3l4lQ0jmzX3b/nApQlXlCs3Q5sXFXvQLPlkMFnOZGIAaQUbT2eDxeV36RemjHqMJhnYyNBoNoZjW1Bt1HTWWA0tVXOWuGUStKXYWqL9LgK8oVPThRbdlTJUGlwbf6/Zj38X38LPFVbxwOwbKPifDnSktASYajF4pxw/Ed8ur+MdUZmfoQaUyqGH44OcYIFmJaUR0/+/QWqKqDyIZioNxVXqUuHc0SSF29Q/ow90uyP2i8CoM/VzRR1HI0Qg7KQVjvdsZ/mLtBPCAZ5kK2KOzDlPZvB/2X/fMvfnCAPSU3fkCbgck8LoE/g2KtvKkQrXoizfDIwb6rxG7J3yAooZU3zvLfbXI+qW+2eU06j/mm6jVN9M8u+ufKPkEkfQpiZs579Yu/lT2XQmlk5dzQ6TFvyKnX32yr/BkrsDAEu0F5HEA+MEbrW4QZuOWV10cNhO8kvtupzgpifBuF6B/SUhUO/VNc+me2hO6F4Y7jWI0YuBkstVukFaukjI22BO22Zr3paBX9pzvTUhwUYgojNL0tLFqxG8YvwGxt+kvEt/4fywwzYw==", "estudio.html": "eNq1W1tv3Eh2ftevoNueadJms29qWeoW5diyfAnktWF5dhYQBLuarO6mRbK4JLvVUk8D+wvyEiDJWzAPCZCHAAHysG95WP+T+QX5CflOFW99kezZyRqwm6w6dc6pcz9V9OE9VzjpdcS1SRr4RzuH9KP5LBzbNZ7Ujg4nnLlHhwFPmeZMWJzw1K5N01FjvwZgORyygNu1mcevIhGnNc0RYcpDgF15bjqxXT7zHN6QL6YXeqnH/EbiMJ/bbcKReqnPj06SdOp6Qus+P2yqkZ3DJL2m334sRLpoNIbj/v12p91t7w8ajYiF3Mf7sO12dvHueyHv3++wjtvdw2s6T/v3+T4/4BxvwRRvB+zAZQxvzHH693vD3nDUwpu4BJbRgbs3XO48XAzFvJF4N1447g9F7PK4gZElycQcCvd6EbB47IX91mDCvfEk7bdbre8GQ+ZcjmMxDd3+jMU6cWoMHOGLOHsHN8ZgBLH027vRXEuuk5QHjalnNlgU+byhBswzPhZc++G1mbAwaSQ89kYDMePxyBdX/Ynnujxc7txPRbSIRAI5irDPhonwpykfYBTMRPOBz0epeoozDvHoeknks+v+yOfzwZhF/X0MMt8bhw0PlJO+A4XxeHDT8EKXz/u9AUE2rmKA0j/LHSvh48UKmo1dS5UYAyW3fps2KnzP1dQsKSifbMTM9aaJ4i1irkvy7kaKtU40V+S04TRNRWhawzRcVKilMcQTsRgsr0gZWi6ot9YoPa4QwrPWBhWlES+cQNDANI0ToIqEJyVxNYFgGiDj8H4oShlkTFlihaX7Hacz6rINpa+s6UN8bOhzdyGA1kuv+1a3l9N1+YhN/ZQWRAuSb7+NZ9ctVR1zn6XejMvho3WhfLsKtvDoupoV8HBaKDgUIR9sGpmyqJYytvY2y/+NNrCH58ALVbDotw9oLmDzRuZte63ZpPQINk2FZN4SEQ/XdjD0hXOJWRrUmKl+lR5WYQYZLXLklM/ThnSLPrnRpg9LAJc7ImZSMkpOGfP7ZFfE8eoGaU8VOcklhZXeYYIF8/0JbXllC2poiwXmi5KA+f5iwzsgN5b2pRoRSfg8usWOxGWhpioOGiaOG1e5PlpAMxRgKdgSk9REX/qaDEs9kjF570jEgfJj2DT/g97AjFGJPiuBRoYi14u5I9GDoWkQboteeWAji1FaPdibXYFDFnpBstgIgqSZ1Tg3+DxNUm903chyWIY5R5Eb0G/2u0qoys2ks+YHXw1Ra0xZ2/hCqtuqSDmuBu6PRiOg8sKR2KJClU2QtKTP7xGPm0ZFFoGkyZWmcy3uDmjfueu2rR6o+HzMQ3dxW2SR6zOrebzm+7u9dd+velVWCDjOXx16Cu9d285KSCz2VuxFc73ZdsvaYqBZ9QC0WmuZI/AWeQTCoqKs2Agj3QovXihFm8W4+3DqbTL1QhRriDEr3G3harvNb+o533x3ueOwcMaS9WB72MwqtsPEib0o1aiutGteQHVhwKLa0SJ7SWr9RS2dxJzX+rVJmkZJv9l03ND6nLjc92axFfK0GUZBUwL9Xctq77WsVnM49XxXjVmBcKc+x4qaqVA1oUkRJs1fh5LPWYAaLGl+ToJmbbnEJiTz2EVT1b5U+OENitY8167BEVC2apoccHyWJHYNSb4mJ6kKTuQ0AJRjai5LWWOGhXxeyxeIsHb0AclkGrPDpoI7WodHiQNBHb3z+A1LCqitqFU8JmCeiORWjCB+XTt6Gjue77MqxsMmNrOxKexz64Trqs36RLRWUMlmkVBqR7/8y5+00+mN9ss//U/JTQUFJaltYvKp65CdQC1vCbbvhgDnMFNPxIDMnjQdpm/cISqswg/cyuFY9jR/vJWEG7MAmd5htaPnePzyMz3fCg1fCAH5jn40PUbfk7BYS5WaS7aUTO+UrMOCYLtg//df/+HftOMvPyOUsF8tXceutcxW7egF1c58+z4A096XUCeofn2X3Qp2IKFOmSs07+aP1h1iB3CjAo2oZt2Ktdsz2+gtu83dW0Fa5n6LzDj2hrdzN0LlXjs6ZlUH+7rgURJtl/vJnOIW1HmnzHMcSgEr9PKfPIxQ0Jb8jVkImSCacV/88qd/34CTWaLwxGKcEvbmqEqeZXyiMVkl3OLKReBKhfDXAxeNxyJdCVpPkXgbGGSbTiMNN4VfHcO64grABtIrFHPwE+F/+c+xCLcELAJKLrlPRvjHKfd5KrbDTMAefOIf/6wds2glmpZRLf9ZSUoqe2DDKiFpDzWWaB9evT850UaxCLS6zBL1QT6/eBsPvfQYKTKGoJZVoDzvONlkcwUU+aeC5eXphxen0DzKt20ofDmVNEuw1eXvhQhOwpkXixAmlm7Fwcv5pLm2QGHbAadJqj2wE/vIFc5UzkDM8fUZRO2kItYTY5BBvXh9enJmLxDF+ud1ZaUfyaLgyNbYH9bNfFDFOe8mn7gwZQLDqkhmsGwwS1U0TLkqG6XMhCGmUpMaXOYs/O718Ym92Ea8X395+kzLRjT9ig9N7aVwRWpqH2SR8DkxSg5RZFqj4RyrXjz7Ax1TaTy3Lk3/IURHbmo/hDFnvqm98VAXCCze0bLlFlr30MXiZ/TLkWzUiHG7CBR7iVchVAIjz1FqEZYYfgbk22d/D6hUphhRIfsRuo95gsqu61pJ6gP07MOplo1++e9Q6z7X9CCosOELt/2x18pYOH37XEOm5rHQ0GWtQHU+tnsVqGB6nUO2e99JHiqq69dVJaLBFLXJFMoDrlKH/fobFjHEdk2OYa6qTDCNclKjvo9haua5XHxElvNCFltBtAuA39OY0qVD+yrAxrDeLmq1Khje//JfEgJS/BhRrmVWFI4B8DpgiJjEiJzU1GR9Wdh9LNXHYzvkV8rlrR/58OXp+2xcX7CQDik9lqBHnXKTJM3jGUcJcIVW4dl0NEKHQVNLuEmOzkK1/c6bc/89nQ3ob1g6sbA/XZ1/ljNmxzAG1UVn6DV0Lwx5/GN2RorHV9JNqujFNI2mCCuoys/oWMpWrJ+9f/msHCwRIwZyaCQCwxnk0+OTsxeeH3jOh3KuwskEZnuFYYuH8pjKph0OdooIQXWwhVVYcDxBHa4XS10EGZ8TUBE2EnQRvCLhM3rXjYEct8rOrQIid6G35uqMt8AUBTEPKmDv3rw/efMSWGJGkSpngvpZAkf4Q2Fty1UWhUdFmdavBUPdMNEA7BqWclueE4RZVskhHkd09DDjxywAUR0VSqmtZkVbhK5tdlqtgncn9SWqlYygg4C5VXYAz4T/HE0JaU6qgIap4X2PtJvybWNnEYe+2lYnJzyOvapoX+L1FfcjGHe7Ze6iFJt3eLetHjqd3Y6TawZJRKfFxRbQbouqq7zhyUSviAfFLn/JRcDT+FrvtLB9w6yoPTMqqs+Zr1dPPpeGPI8SsUXlBLWu1txuSLd597rZyedQn0P0Co/aesmoBCk4nfDAqzD6Cq9JNIGQT0k5MKyR/IMtd/Fnd9e09qq7puWl+U7DCqrn+eET89dwdYBhGloOS9IVFjGm/MlC30sOTo4O8ezum/RPlS5gC7KxF9xNdn+fMZBtVxFgERDsILtozhSqmvq+GSDcZI9MokjsxRJZFn161ePoHW4wFPPK6DMx72LwFexp36QqTFmOwkZlJ6FiM+Yp2z7DxooNqAJGjpc1DE3vjKahZESTXaN+aSxQoalFp/Yia/r656VkzfOOuWt2L8xy19CY2bZaJsn4wsz7v2xRx3XNroVlPRNCpmXDkbuPZVbPtA5oqVpG1WjRBKq1rLUHwwXEeWMXhOXivdbjFtEERhir/KvWFz1hhdkuaJx3zZ7ZaNPa0ajXe/zYJPlZYLrdM0fMT7BWdoiVddaued7Csp5apcwT6/YkXbVoeX55MQDT0s4oRJIpveJz/fS8daHMj84DwwR+ZZ+ety/kUH4cJO3OsqzT886FYQVTP/Ui//rMYT6L9VdIRF6wjrV7oYarWHclCzS6grjRefjKpL+N7kMgIxdaWda7yOy0UpRi+PHFExWm+2RTmyCvKyj2JGVvpN+jdcZiC41H1t5yZ1la2BD2rBsLOIM1jdAPItelsTf/UcRIWqREOgicwzMv+UkQpdd6njuGG15AtAkP+pkZEgHXhX20ADPC8hKKhEb+cnZJicBVY8Iit+F0KqsYACIBMQdI4PwZpT0qI4jJwRDD0TVQDMthg7Ksf6243sVcUPJvLGU3xWEZIDPOAu9XcBdg30yE5DMNIUx9aCyXVK8gQ0KyYl6R8wiewHV2Y8M3uG+3O9KnlfQziTo2YRrz9FgeKlYSx+9lowERG2aSA8lKaBvI4JWt6ik21xPr2kSeVaqRRNzq5NwkgMS6MR62rYOmnElZSEnXGonZwzzBoII0kGUKJMxmN8UknYBwm/vVAQnK1hzAseaPJAyKfJ0ZCt4Ric6Nh67pWNflbDZy86iAWYdXW0JiR3SCNJT26KAeVEPOYtttoriQryN6e9hTL8rG3sXis0oXSqe6qicKA6x6CLUeCVyEckZqt8yZ3RrcbeZZWWGXxjSYPbLHFktBCy03T0rBOLCydJDStDwxfpL9qolmt3/HsmaXzK3QyjeYBmAf6HV5gVE3LFWOfXhzan96sJDSlWWmnqLME6eCPjs4A+lwrBtLDQ9ffg7HU18k2l/+rD1YzLYAzb78R4xwz5PDYXz0YAELA9AL5FdX72D6yz9rNHi9bfBmZTD49Ih8VydOLdlAJU8+KbqVoaXqrZJP/Xodzs6S69DRCsXxOXqaRB8ZC2ggU0pssyvmoVLjqTPRR+YC6pkIt19/dfL0eb10XrQQl0uHERA3FtmgzDHL5c46oWQirnQ6za7kaYeFbmLLA4FzmroYkP2MPJ/L2mCA7k7PikZNjBS4ASuStYI1YcS3sZDwo8EQffblMgvtNGYojnJ9UiUBfcobBSu7aLDrdIdRL81jnG1dlR0W/TylfegSYZY4YNjGQuUY9ARixuXIEjNlcbMGUJkYrFVAS5URYNtyxVfcRmyUhmJbSSvQqUxRAgXHoIBKXmplICM90JGsbduu0yFJ3ZCRWpbTa1X5GcKci7qmrLbVFQ4KoYOh4+yZ8ITxJORJ0rd6PZO+3fHVW6unfG6FWnZQU6dNfDNFiAHdlOzksua5QvTxErlEUSqrV9LFYD1tBKH0e7TP1rWUcBEkrht2EGa9QTmI8EVNy+oIkJZl8KAsgwdkXvJQdCVe1OuZvSA60SGEBLd8Ho5TKDLDVGz/aQ7yhiaKXeSGKa/u7PP6ew6W6KDkWJ110JOI0ffh4Yz5U5dlT6l8eAZHkQ/vRMI/fqhfKJwrHCUiTnWdmUPDPtIlHRVd38LRLPoWy3h0cGB8d3DQWJsdrsxm7JY+65DPVikZi0xm545ceWFLIViO70VP5QwlJ4kl3/bQLk4LHPh3yrPWVq+rU9k6FSXUbh9n34gpzHAJuLfD9fpH7F2TUCIEHXQrOrZJvq8rUGNFe9XTiGFWGkngnPNcARdP8qd+dYuooDOsamtZsqQ+XmKj+Ia4dYYQkPnkbe5erf/yFdkJETFMQQT85l8A2fcymOqkPGU/RXzPo1BdCoy8hac/ohfUCZjOzleA6diZeWGiwIt8qO4NNiLounf/9FM+Ik9r60/q8jK33q/ThXd9A1vpL+uYntQPh8X94xEly2PmMk3GIM3WpqFKbBbNPBewSWxLC/gNdhKams80+mSF0aArhj6z6n2pznX2iEh+Iyqp1B+pjKqOKn/6CWnzUV1TbybIssqBZV85uSpcjYFqRaHSU3XYlpVFg9vzTyGU7efmT30fS+UlsJZbPKrO+IQh6Q7to2FFb6kYj32lZHNo0bUV1GzNsFeZdmXBlvX0FGxUFCtLOGnmYV6YoXrNLD68GFBGZXk6VQmQMHz/ffZwz7aZkT2jknT52yl6xK4xYBaddKKdl6OvQxq0JCFjkLPBvrb76kcp37r7akiw7XAjHhhGtXjNvEGEsseT27ojBetP45hd41X+6mUyM56Uz/3z8vmiZDqwjwKLPE5ajA2KyF/L4qxFmo1dzw4v6pWz0k255B/trYskj3McfHOYm4hQyEdsLANUkRBde2ipTx6ziJqNC9vdHgoQF1XsuJulCjdz+2i+LQZJTIY0KmFUqVH2zqYpqReEMH4ywwMB0TGtXpc7rJsUyv+f2DHulPW5upG/uE3WxMii0F/pfP56RFh+Cx3nK3RWT29VGqEIE4v0zqjvUcrOWXMoBtKdNuqxX91hZ6CpvTmJBtZUhyFz1FavHlqtx9Sk5sXMer+bAjxFf52ijwVwr6eazGrDmq71nSoOLYua/ZyZKCUqO7MQWz2UCCakEbBI/900GFLhnR0vmPxrargz4FZ1IZuaSqwlMyo1cZfW7q2+b9VfNaStghtLRcdJ4zU6eSbaqaT2VT6Um4f2vW9I/oOtMFW+EMEGlfiZ8ZUVIHcQ3ihRthDegFknrGyawna1pyrvCrIzs+yiQpbUJWARni2XR+nkA4Ju5kvbYOj7OPXfDzrVo/ZKW6fs+2+ZORaBVflUXTV6gZXdgNjyBkRmk+w071sb0MHfmumcw/ZgdQPqqEDynNvNZNNxiost9aDLbZkIJHkUYrf2CKxOJcgE2dbecj0Gc3oOx/3h/ale9+iOt0n3vbQCbXRI5Ro6ZPURiLwJxoRkSycr38xJdHt+w1VSWlCcY/Kaz95+tXf3Odu33uauXxcXTeSpEJFecTsXnRGV4RTXn3M0h3rWoXz//Uqj4m4G29ukT7R3dHnKIwnB7OgISd7nqEMUvXqMVI+98UdZV39OZF8hfyHL4gCpjOio24uboLdDEo41Q2/LE1WYs9iZeDOB0nyxVP3J2kHRr/uuxKx+lFD9AmH1i4MLA2ypW8JtNDM+L/l1otP3JcYauJIFdB14CYIH0sy5ZdHvhcxR2WmZcj4FWxzNGeq8S+HKdvzXZq+yYVTHbpXkdWElArkDPKycr1UOTwObnDT7HK3+bVLYOK77FU47qvoivZTNIiE/H108+qQdyv+acPRgMcrzvoVNR7B/g769lZOfEHmqrT2TUXKNd8km8XsvQ/7beK22IKOt9JWaZQ1RBzQJdGlQ7i4+GUZLSt8KHzbV/6X7PwSGMIY="}""")
for _n, _d in _FILES.items():
    open(f"{APP}/ap3d/{_n}", "wb").write(zlib.decompress(base64.b64decode(_d)))
if APP not in sys.path:
    sys.path.insert(0, APP)
print(subprocess.run("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader",
                     shell=True, capture_output=True, text=True).stdout.strip() or "SIN GPU: cambia el entorno a GPU")
import psutil
print(f"RAM del sistema: {psutil.virtual_memory().total/2**30:.0f} GB · CPU: {os.cpu_count()}")
print("Proyecto listo en", APP)

## 2. Instalar dependencias y Blender 4.2 LTS

In [ ]:
# @title 2. Instalar dependencias, Hunyuan3D-2 y Blender 4.2 LTS
USAR_CACHE_EN_DRIVE = False  # @param {type:"boolean"}
# @markdown Si lo activas, las piezas compiladas se guardan en Drive y la próxima sesión se instalan en segundos.
import os, subprocess, sys, glob, shutil, time
t0 = time.time()

def sh(cmd, check=True):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stdout[-3000:], r.stderr[-3000:])
        if check:
            raise RuntimeError("Fallo: " + cmd)
    return r.stdout

# --- sistema (Blender sin ventana necesita estas librerias)
sh("apt-get -qq update && apt-get -qq install -y libxi6 libxkbcommon0 libsm6 libxrender1 "
   "libxxf86vm1 libxfixes3 libgl1 libegl1 libglu1-mesa libopengl0 > /dev/null", check=False)

# --- Python
sh(f"{sys.executable} -m pip install -q 'gradio>=5.9,<6' 'diffusers==0.32.2' 'transformers==4.49.0' 'accelerate>=1.2,<1.6' "
   "einops omegaconf trimesh pymeshlab pygltflib xatlas rembg onnxruntime-gpu "
   "opencv-python-headless ninja pybind11 psutil hf_transfer 'huggingface_hub>=0.25'")
print(f"Paquetes listos ({time.time()-t0:.0f}s)")

# --- Hunyuan3D-2
if not os.path.exists("/content/Hunyuan3D-2"):
    sh("git clone -q --depth 1 https://github.com/Tencent/Hunyuan3D-2 /content/Hunyuan3D-2")
sh(f"cd /content/Hunyuan3D-2 && {sys.executable} -m pip install -q -e . --no-deps")

import torch
cap = "%d.%d" % torch.cuda.get_device_capability(0) if torch.cuda.is_available() else "8.0"
os.environ["TORCH_CUDA_ARCH_LIST"] = cap            # compila solo para tu GPU: mas rapido
os.environ["MAX_JOBS"] = str(os.cpu_count())
tag = f"py{sys.version_info.major}{sys.version_info.minor}_torch{torch.__version__.split('+')[0]}_sm{cap}"
cache = None
if USAR_CACHE_EN_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    cache = f"/content/drive/MyDrive/ap3d_cache/{tag}"
    os.makedirs(cache, exist_ok=True)

def have(mod):
    return subprocess.run([sys.executable, "-c", f"import {mod}"], capture_output=True).returncode == 0

if not have("custom_rasterizer"):
    whl = glob.glob(f"{cache}/custom_rasterizer*.whl") if cache else []
    if whl:
        sh(f"{sys.executable} -m pip install -q {whl[0]}")
        print("custom_rasterizer desde la cache de Drive")
    else:
        print("Compilando custom_rasterizer (3-5 min)...")
        d = "/content/Hunyuan3D-2/hy3dgen/texgen/custom_rasterizer"
        sh(f"cd {d} && {sys.executable} -m pip wheel -q . --no-build-isolation --no-deps -w /content/whl")
        w = glob.glob("/content/whl/custom_rasterizer*.whl")[0]
        sh(f"{sys.executable} -m pip install -q {w}")
        if cache:
            shutil.copy(w, cache)

d = "/content/Hunyuan3D-2/hy3dgen/texgen/differentiable_renderer"
if not glob.glob(f"{d}/mesh_processor*.so"):
    so = glob.glob(f"{cache}/mesh_processor*.so") if cache else []
    if so:
        shutil.copy(so[0], d)
    else:
        sh(f"cd {d} && c++ -O3 -Wall -shared -std=c++11 -fPIC "
           f"$({sys.executable} -m pybind11 --includes) mesh_processor.cpp "
           f"-o mesh_processor$({sys.executable}-config --extension-suffix 2>/dev/null || "
           f"python3-config --extension-suffix)", check=False)
        if not glob.glob(f"{d}/mesh_processor*.so"):
            sh(f"cd {d} && {sys.executable} setup.py build_ext --inplace", check=False)
        if cache:
            for f in glob.glob(f"{d}/mesh_processor*.so"):
                shutil.copy(f, cache)
print(f"Hunyuan3D-2 listo ({time.time()-t0:.0f}s)")

# --- Blender 4.2 LTS
if not os.path.exists("/content/blender/blender"):
    url = "https://download.blender.org/release/Blender4.2/blender-4.2.3-linux-x64.tar.xz"
    sh(f"wget -q {url} -O /content/blender.tar.xz && tar -xf /content/blender.tar.xz -C /content "
       "&& mv /content/blender-4.2.3-linux-x64 /content/blender && rm /content/blender.tar.xz")
print(sh("/content/blender/blender -b --version | head -1").strip())
print(f"Instalación lista en {time.time()-t0:.0f}s")

## 3. Comprobar GPU, Blender y modelos de IA (carga todo en RAM y GPU)

In [ ]:
# @title 3. Comprobar GPU, descargar modelos, precargarlos en la RAM y cargarlos en la GPU
CARGAR_MULTIVISTA = True  # @param {type:"boolean"}
CARGAR_TEXTURA = True  # @param {type:"boolean"}
import os, sys, time
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
sys.path.insert(0, "/content/ap3d_app")
sys.path.insert(0, "/content/Hunyuan3D-2")
# Hunyuan3D-2 necesita transformers 4.49: con versiones nuevas su codificador
# de imagen (DINOv2) no carga ("Missing key(s) ... main_image_encoder").
import importlib.metadata as _md, subprocess
if _md.version("transformers") != "4.49.0" or not _md.version("diffusers").startswith("0.32"):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers==4.49.0",
                    "diffusers==0.32.2", "accelerate>=1.2,<1.6"], check=True)
    if "transformers" in sys.modules:
        raise SystemExit("Versiones corregidas. Ve a Entorno de ejecución › Reiniciar sesión "
                         "y vuelve a ejecutar la celda 1 y después esta celda 3.")
import torch
from huggingface_hub import snapshot_download
from ap3d import config
from ap3d.engine import get_engine

print("CUDA:", torch.cuda.is_available())
P = config.detect()
print(config.describe(P))

t = time.time()
pats = ["hunyuan3d-dit-v2-0/*"]
if CARGAR_TEXTURA:
    pats += ["hunyuan3d-paint-v2-0/*", "hunyuan3d-delight-v2-0/*"]
snapshot_download("tencent/Hunyuan3D-2", allow_patterns=pats, max_workers=16)
if CARGAR_MULTIVISTA:
    snapshot_download("tencent/Hunyuan3D-2mv", allow_patterns=["hunyuan3d-dit-v2-mv/*"], max_workers=16)
print(f"Modelos descargados ({time.time()-t:.0f}s)")

eng = get_engine()
if P["preload_weights"]:
    eng.preload_weights()          # deja todos los pesos en la RAM
eng.load(want_mv=CARGAR_MULTIVISTA, want_paint=CARGAR_TEXTURA)
print("Todo listo. Sigue con la celda 4 o 5.")

## 4. Estudio

In [ ]:
# @title 4. Estudio (dentro del notebook)
import sys; sys.path.insert(0, "/content/ap3d_app")
from ap3d.app import launch_inline
launch_inline(port=7860)

## 5. Abrir la página web con contraseña

In [ ]:
# @title 5. Abrir la página web con contraseña (para el celular)
CONTRASENA = ""  # @param {type:"string"}
# @markdown Déjala vacía para generar una al azar. El usuario es **ap**. La página deja de funcionar al cerrar la sesión de Colab.
import sys; sys.path.insert(0, "/content/ap3d_app")
from ap3d.app import launch_web
url, user, pw = launch_web(CONTRASENA, user="ap", port=7861)
print("Página web:", url)
print("Usuario:", user, "· Contraseña:", pw)

## 6. Guardar en Drive

In [ ]:
# @title 6. Guardar todos los resultados en Google Drive
CARPETA = "AP_Model_3D"  # @param {type:"string"}
import os, shutil, glob
from google.colab import drive
if not os.path.exists("/content/drive/MyDrive"):
    drive.mount("/content/drive")
dst = f"/content/drive/MyDrive/{CARPETA}"
os.makedirs(dst, exist_ok=True)
n = 0
for z in glob.glob("/content/ap3d_salidas/*.zip"):
    if not os.path.exists(os.path.join(dst, os.path.basename(z))):
        shutil.copy(z, dst); n += 1
print(f"{n} modelos nuevos copiados a {dst}")

## 7. Memoria

In [ ]:
# @title 7. (Opcional) Liberar GPU / ver uso de memoria
LIBERAR_MODELOS = False  # @param {type:"boolean"}
import gc, torch, psutil, sys
sys.path.insert(0, "/content/ap3d_app")
from ap3d.engine import get_engine
e = get_engine()
if LIBERAR_MODELOS:
    e.shape = e.shape_mv = e.paint = None; e.ready = False
    gc.collect(); torch.cuda.empty_cache()
e.log_mem()